# Sistema de Forecast Predictivo de Fallas — V6
## Planta de Motores Silao — Volkswagen de México

**Flujo de ejecución (una sola pasada `Run All`):**
1. Configuración global
2. Imports y utilidades
3. Carga de datos (inicial o incremental)
4. Agregación mensual por serie
5. Análisis de cohortes y curva de hazard
6. Feature engineering (sin data leakage)
7. Validación estadística de variables (FDR)
8. Optimización Optuna (si `REOPTIMIZAR_HIPERPARAMETROS=True`)
9. Backtesting walk-forward (≥6 folds, WAPE OOS ≤ 11%)
10. Modelo global XGBoost
11. Modelos individuales por serie
12. Forecast recursivo 6 meses con cap
13. Isolation Forest (anomalías multivariadas)
14. CUSUM + detección estadística
15. Score 7 componentes
16. Watchlist Top 50
17. Exportar 8 CSVs para Power BI
18–26. Visualizaciones
27. Checklist anti-overfitting
28. Resumen y rollback


In [1]:
# ── CONFIGURACIÓN GLOBAL ─────────────────────────────────────────────────────
from pathlib import Path

# Archivos de datos
DATA_FILE             = "All_Engines_Silao.xlsx"
NUEVO_DATA_FILE       = "Datos_Power_BI_-_Silao_NUEVO.xlsx"
VOLUMEN_FILE          = "volumen_produccion.xlsx"

# Directorios de salida
MODEL_DIR  = Path("modelos_ml")
OUTPUT_DIR = Path("outputs")

# Parámetros de forecast
HORIZONTE_MESES          = 6
MIN_FALLAS_SERIE         = 3
MIN_MESES_SERIE          = 6
MIN_MESES_ENTRENAMIENTO  = 12
WF_FOLDS_MINIMOS         = 6

# Umbrales de aceptación del modelo
WAPE_MAX_OOS          = 0.11    # ≤ 11 %
GAP_OVERFITTING_MAX   = 0.15    # gap (in-sample – OOS) ≤ 15 pp
COBERTURA_CI_MIN      = 0.88    # cobertura CI 95 % ≥ 88 %

# Regularización XGBoost (no negociable)
XGB_PARAMS_BASE = dict(
    reg_alpha        = 1.0,
    reg_lambda       = 2.0,
    min_child_weight = 3,
    max_depth        = 5,
    subsample        = 0.85,
    colsample_bytree = 0.9,
    n_estimators     = 300,
    learning_rate    = 0.05,
    random_state     = 42,
    objective        = "reg:squarederror",
    verbosity        = 0,
)

# Pesos del score de alerta (7 componentes, deben sumar 1.0)
PESOS_SCORE = {
    "desviacion_stat"    : 0.20,   # z-score Repair Date
    "aceleracion_cusum"  : 0.18,   # CUSUM
    "senal_engine_date"  : 0.15,   # cohorte Engine Date (señal anticipada)
    "anomalia_if"        : 0.17,   # Isolation Forest
    "error_forecast_prev": 0.10,   # error forecast previo
    "crecimiento_proy"   : 0.12,   # crecimiento proyectado
    "costo_relativo"     : 0.08,   # costo relativo al total
}
assert abs(sum(PESOS_SCORE.values()) - 1.0) < 1e-9, "Los pesos del score deben sumar exactamente 1.0"

# Optuna
N_OPTUNA_TRIALS              = 50
REOPTIMIZAR_HIPERPARAMETROS  = False   # True sólo en el primer run o cuando se desee re-optimizar

# Decay temporal en pesos de entrenamiento
DECAY_RATE = 2.5

# Paleta corporativa
COLOR_NAVY   = "#0A2540"
COLOR_TEAL   = "#1E5F74"
COLOR_GOLD   = "#D4A74A"
COLOR_RED    = "#C0392B"
COLOR_ORANGE = "#E67E22"
COLOR_YELLOW = "#F39C12"
COLOR_GREEN  = "#16A085"
COLOR_PURPLE = "#8B4789"
COLOR_LIGHT  = "#F7F5F0"

# Extensibilidad: agregar variables nuevas aquí
CANDIDATAS_NUEVAS_CATEGORICAS: list = []
CANDIDATAS_NUEVAS_NUMERICAS:   list = []

print("✅ Configuración cargada")


✅ Configuración cargada


In [2]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import logging
import json
import gc
import hashlib
from datetime import datetime
import joblib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, f_oneway, kruskal

import optuna
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("silao_v6")

# Crear directorios
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_DIR / "individuales").mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Imports OK  |  pandas {pd.__version__}  |  numpy {np.__version__}")


✅ Imports OK  |  pandas 3.0.3  |  numpy 2.4.4


In [3]:
# ── FUNCIONES UTILITARIAS ────────────────────────────────────────────────────

MESES_ES = {
    "enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,
    "julio":7,"agosto":8,"septiembre":9,"octubre":10,"noviembre":11,"diciembre":12
}


def parse_fecha(s) -> pd.Timestamp:
    """Parse fecha en español tipo 'lunes, 14 de marzo de 2024'."""
    if pd.isna(s):
        return pd.NaT
    try:
        partes = str(s).lower().split(", ", 1)[-1].split(" de ")
        return pd.Timestamp(int(partes[2]), MESES_ES[partes[1]], int(partes[0]))
    except Exception:
        return pd.NaT


def wape(actual: np.ndarray, pred: np.ndarray) -> float:
    """Weighted Absolute Percentage Error. NaN si sum(actual)==0."""
    actual = np.asarray(actual, dtype=float)
    pred   = np.asarray(pred,   dtype=float)
    total  = float(np.nansum(np.abs(actual)))
    if total == 0:
        return np.nan
    return float(np.nansum(np.abs(actual - pred)) / total)


def fdr_correction(p_values: np.ndarray, alpha: float = 0.05):
    """Benjamini-Hochberg FDR. Returns (rejected_mask, adjusted_p_values)."""
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    if n == 0:
        return np.array([], dtype=bool), np.array([], dtype=float)
    idx   = np.argsort(p)
    p_s   = p[idx]
    p_adj = np.minimum.accumulate(p_s[::-1] * n / np.arange(n, 0, -1))[::-1]
    out   = np.empty(n)
    out[idx] = p_adj
    return out < alpha, out


def cap_forecast(pred: float, historia_real: np.ndarray) -> float:
    """
    Aplica cap conservador al forecast recursivo.
    historia_real contiene SÓLO observaciones reales, nunca predicciones.
    """
    if len(historia_real) == 0:
        return max(0.0, float(pred))
    max_h  = float(np.max(historia_real))
    tail   = historia_real[-min(6, len(historia_real)):]
    mean_r = float(np.mean(tail))
    cap_hi = min(max_h * 1.5, mean_r * 2.5)
    cap_lo = max(0.0, mean_r * 0.3)
    return float(np.clip(pred, cap_lo, cap_hi))


def sample_weights_decay(n: int, decay: float = DECAY_RATE) -> np.ndarray:
    """Decay exponencial: meses recientes tienen más peso en entrenamiento."""
    t = np.linspace(0, 1, n)
    w = np.exp(t * decay)
    return (w / w.mean()).astype(float)


def ci_bandas(pred: np.ndarray, residuals: np.ndarray):
    """
    CI 95 % que se ensancha con el horizonte.
    El ancho crece como sqrt(h) para reflejar mayor incertidumbre en M+6.
    """
    sigma = float(np.nanstd(residuals)) if len(residuals) > 1 else float(np.nanmean(np.abs(residuals)) + 1e-8)
    z     = 1.96
    h     = np.arange(1, len(pred) + 1)
    lo    = np.maximum(pred - z * sigma * np.sqrt(h), 0)
    hi    = pred + z * sigma * np.sqrt(h)
    return lo, hi


def cusum_stat(series: np.ndarray, k: float = 0.5) -> float:
    """CUSUM acumulado positivo normalizado por sigma del baseline."""
    if len(series) < 3:
        return 0.0
    half    = max(3, len(series) // 2)
    mu      = float(np.mean(series[:half]))
    sigma   = float(np.std(series[:half])) + 1e-8
    slack   = k * sigma
    s       = 0.0
    for x in series:
        s = max(0.0, s + (x - mu) - slack)
    return s / sigma


def slope_lineal(arr: np.ndarray) -> float:
    """Pendiente OLS sobre los últimos puntos."""
    if len(arr) < 2:
        return 0.0
    x = np.arange(len(arr), dtype=float)
    return float(np.polyfit(x, arr.astype(float), 1)[0])


def hash_serie(serie_id: str) -> str:
    """Hash corto para usar como nombre de archivo de modelo individual."""
    return hashlib.md5(serie_id.encode()).hexdigest()[:12]


print("✅ Utilidades cargadas")


✅ Utilidades cargadas


In [4]:
# ── CARGA DE DATOS Y MODO DE EJECUCIÓN ───────────────────────────────────────

hay_datos_nuevos = Path(NUEVO_DATA_FILE).exists()
hay_modelo_previo = (MODEL_DIR / "global_xgb.pkl").exists()
MODO = "INCREMENTAL" if (hay_datos_nuevos and hay_modelo_previo) else "INICIAL"

print(f"📂 Modo: {MODO}  |  datos_nuevos={hay_datos_nuevos}  |  modelo_previo={hay_modelo_previo}")

# ── Cargar dataset principal ─────────────────────────────────────────────────
df_raw = pd.read_excel(DATA_FILE, sheet_name="923-All Engines Silao")
print(f"\n📊 Dataset cargado: {len(df_raw):,} filas × {len(df_raw.columns)} columnas")

# Asegurar tipos fecha
_fecha_cols = ["Production Date", "Engine Date", "SALES DATE", "LOADING DATE", "Repair date"]
for col in _fecha_cols:
    if col in df_raw.columns and df_raw[col].dtype == object:
        df_raw[col] = pd.to_datetime(df_raw[col], errors="coerce")

# ── Modo incremental: combinar con nuevos datos ───────────────────────────────
if MODO == "INCREMENTAL":
    df_nuevo = pd.read_excel(NUEVO_DATA_FILE)
    cols_c = [c for c in df_raw.columns if c in df_nuevo.columns]
    df_nuevo = df_nuevo[cols_c].copy()
    for col in _fecha_cols:
        if col in df_nuevo.columns and df_nuevo[col].dtype == object:
            df_nuevo[col] = pd.to_datetime(df_nuevo[col], errors="coerce")
    n_antes = len(df_raw)
    df_raw = (pd.concat([df_raw, df_nuevo], ignore_index=True)
                .drop_duplicates(subset=["APPLICATION NO"], keep="last"))
    print(f"   ➕ Filas netas añadidas: {len(df_raw) - n_antes:,}")

# ── Limpieza básica ───────────────────────────────────────────────────────────
df_raw = df_raw.dropna(subset=["Repair date", "EA-Number", "DAMAGE CODE", "Basic no. Description"])
df_raw["COSTS"] = pd.to_numeric(df_raw["COSTS"], errors="coerce").fillna(0).clip(lower=0)
df_raw["MIS"]   = pd.to_numeric(df_raw["MIS"],   errors="coerce").fillna(0).clip(lower=0)
df_raw["KM"]    = pd.to_numeric(df_raw["KM"],    errors="coerce").fillna(0).clip(lower=0)

# ── Construir serie_id DESPUÉS de cargar datos ────────────────────────────────
# Nota: se construye en tiempo de ejecución, nunca hardcodeado
df_raw["serie_id"] = (
    df_raw["EA-Number"].astype(str).str.strip() + "||" +
    df_raw["Basic no. Description"].astype(str).str.strip() + "||" +
    df_raw["DAMAGE CODE"].astype(str).str.strip()
)

# Columnas de período
df_raw["mes_repair"] = df_raw["Repair date"].dt.to_period("M")
df_raw["mes_engine"] = df_raw["Engine Date"].dt.to_period("M")

# ── Excluir último mes de Repair Date (puede estar incompleto) ───────────────
max_mes_repair = df_raw["mes_repair"].max()
df_work = df_raw[df_raw["mes_repair"] < max_mes_repair].copy()
print(f"\n📅 Rango Repair Date: {df_work['mes_repair'].min()} → {df_work['mes_repair'].max()}")
print(f"   Mes excluido (incompleto): {max_mes_repair}")
print(f"   Filas activas: {len(df_work):,}")
print(f"   Series únicas: {df_work['serie_id'].nunique():,}")

# ── Volumen de producción (opcional) ────────────────────────────────────────
hay_volumen = Path(VOLUMEN_FILE).exists()
vol_df = None
if hay_volumen:
    vol_df = pd.read_excel(VOLUMEN_FILE)
    vol_df.columns = [c.lower().strip() for c in vol_df.columns]
    vol_df["cohorte"] = pd.to_datetime(vol_df["cohorte"], errors="coerce").dt.to_period("M")
    print(f"\n✅ volumen_produccion.xlsx cargado → tasa de falla = fallas / motores producidos")
else:
    print("\nℹ️  Sin volumen_produccion.xlsx → tasa de falla como proxy por distribución de edad")


📂 Modo: INICIAL  |  datos_nuevos=False  |  modelo_previo=True



📊 Dataset cargado: 49,361 filas × 76 columnas

📅 Rango Repair Date: 2024-01 → 2025-11
   Mes excluido (incompleto): 2025-12
   Filas activas: 27,830
   Series únicas: 890

ℹ️  Sin volumen_produccion.xlsx → tasa de falla como proxy por distribución de edad


In [5]:
# ── AGREGACIÓN MENSUAL POR SERIE ────────────────────────────────────────────

def agregar_mensual(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye el panel mensual: una fila por (serie_id × mes_repair).
    Calcula fallas (conteo), costo_total, mis_mean, km_mean.
    """
    grp = df.groupby(["serie_id", "mes_repair"])
    agg = grp.agg(
        fallas      = ("APPLICATION NO", "count"),
        costo_total = ("COSTS",          "sum"),
        mis_mean    = ("MIS",            "mean"),
        km_mean     = ("KM",             "mean"),
    ).reset_index()

    # Metadatos de la serie (estáticos – tomar la moda)
    def safe_mode(x): return x.mode().iloc[0] if len(x) and len(x.mode()) else "UNK"
    def mis_cl(x):
        cut = pd.cut(x, bins=[0,3,7,12,24], labels=["0-3","4-7","8-12","13-24"], right=True)
        m = cut.dropna().mode()
        return str(m.iloc[0]) if len(m) else "0-3"
    meta = df.groupby("serie_id").agg(
        ea_number   = ("EA-Number",              safe_mode),
        componente  = ("Basic no. Description",  safe_mode),
        damage_code = ("DAMAGE CODE",            safe_mode),
    ).reset_index()
    meta["mis_cluster"] = (
        df.groupby("serie_id")["MIS"].apply(mis_cl).values
    )

    monthly = agg.merge(meta, on="serie_id", how="left")
    monthly["mes_dt"] = monthly["mes_repair"].dt.to_timestamp()
    monthly = monthly.sort_values(["serie_id", "mes_repair"]).reset_index(drop=True)
    return monthly


monthly = agregar_mensual(df_work)
print(f"✅ Panel mensual: {len(monthly):,} filas  ({monthly['serie_id'].nunique()} series × {monthly['mes_repair'].nunique()} meses)")

# Estadísticas de series
n_por_serie = monthly.groupby("serie_id")["fallas"].agg(["sum","count"])
n_por_serie.columns = ["total_fallas","n_meses"]
print(f"\nDistribución de series por meses con datos:")
print(n_por_serie["n_meses"].describe().round(1).to_string())
print(f"\nSeries con ≥ {MIN_MESES_SERIE} meses:  {(n_por_serie['n_meses'] >= MIN_MESES_SERIE).sum()}")
print(f"Series con ≥ {MIN_MESES_ENTRENAMIENTO} meses: {(n_por_serie['n_meses'] >= MIN_MESES_ENTRENAMIENTO).sum()}")

# Costo unitario de referencia por serie (mediana histórica)
costos_ref = (
    df_work[df_work["COSTS"] > 0]
    .groupby("serie_id")["COSTS"]
    .agg(costo_unit_med="median", costo_unit_p75=lambda x: x.quantile(0.75))
    .reset_index()
)


✅ Panel mensual: 4,813 filas  (890 series × 23 meses)

Distribución de series por meses con datos:
count    890.0
mean       5.4
std        5.8
min        1.0
25%        1.0
50%        2.5
75%        8.0
max       23.0

Series con ≥ 6 meses:  289
Series con ≥ 12 meses: 154


In [6]:
# ── ANÁLISIS DE COHORTES Y CURVA DE HAZARD ──────────────────────────────────
# Engine Date = señal anticipada con CENSURA (lotes recientes aún no tuvieron tiempo de fallar)

def calcular_hazard_por_familia(df: pd.DataFrame, max_mis: int = 23) -> dict:
    """
    Curva de hazard empírica por EA-Number.
    Cuando hay volumen_produccion.xlsx → tasa real = fallas / motores producidos.
    Sin él → proxy = P(falla|MIS=k) ≈ fallas_k / fallas_k+
    Sólo incluye cohortes con ≥ 3 meses de exposición y ≥ 3 fallas observadas.
    """
    hazard = {}
    hoy = df["mes_repair"].max()

    for ea, grp in df.groupby("EA-Number"):
        # Filtrar cohortes confiables (≥3 meses de exposición desde Engine Date)
        grp = grp.copy()
        grp["meses_expuesto"] = (hoy - grp["mes_engine"]).apply(
            lambda x: x.n if pd.notna(x) else 0
        )
        # Excluir cohortes con < 3 meses de exposición
        grp_conf = grp[grp["meses_expuesto"] >= 3]

        if hay_volumen and vol_df is not None:
            # Tasa real: fallas / motores producidos en cohorte
            vol_ea = vol_df[vol_df.get("ea_number", vol_df.get("codigo_motor", vol_df.columns[1])) == ea]
            mis_counts = grp_conf.groupby("MIS")["APPLICATION NO"].count()
            fallas_tot = max(mis_counts.sum(), 1)
            # Hazard condicional por MIS
            h = {}
            for k in range(max_mis + 1):
                at_risk = mis_counts[mis_counts.index >= k].sum()
                h[k] = float(mis_counts.get(k, 0)) / max(at_risk, 1)
        else:
            # Proxy: distribución relativa por MIS
            mis_counts = grp_conf.groupby("MIS")["APPLICATION NO"].count()
            total = max(mis_counts.sum(), 1)
            h = {}
            for k in range(max_mis + 1):
                at_risk = mis_counts[mis_counts.index >= k].sum()
                h[k] = float(mis_counts.get(k, 0)) / max(at_risk, 1)

        hazard[ea] = h

    return hazard


hazard_curves = calcular_hazard_por_familia(df_work)
print("✅ Curvas de hazard calculadas para familias:", list(hazard_curves.keys()))

def calcular_cohorte_ratio(df: pd.DataFrame, hazard: dict) -> pd.DataFrame:
    """
    Para cada (serie_id × mes_engine), calcula:
      cohorte_activa_ratio = fallas_obs / fallas_esperadas_por_hazard
    Solo cohortes con ≥ 3 meses exposición y ≥ 3 fallas.
    Los últimos 2 meses de Engine Date se marcan como 'no_confiable'.
    """
    hoy = df["mes_repair"].max()
    max_mes_engine = df["mes_engine"].max()
    # Umbral: cohortes más recientes que este mes se excluyen del ratio
    umbral_reciente = max_mes_engine - 2   # últimos 2 meses excluidos

    rows = []
    for (sid, mes_e), g in df.groupby(["serie_id", "mes_engine"]):
        ea = g["EA-Number"].iloc[0]
        meses_exp = (hoy - mes_e).n if hasattr((hoy - mes_e), "n") else 0

        if meses_exp < 3 or len(g) < 3:
            continue

        confiable = (mes_e <= umbral_reciente)
        hz = hazard.get(ea, {})
        # Fallas esperadas = sum(hazard[0..meses_exp]) — proporcional a tamaño cohorte
        fallas_obs = len(g)
        expected_cumul = sum(hz.get(k, 0) for k in range(min(meses_exp + 1, 24)))
        proxy_vol = fallas_obs / max(expected_cumul, 1e-8)  # estimación del tamaño de cohorte
        fallas_esperadas = expected_cumul * proxy_vol

        ratio = fallas_obs / max(fallas_esperadas, 1e-8)

        rows.append({
            "serie_id":            sid,
            "mes_engine":          mes_e,
            "cohorte_ratio":       ratio if confiable else np.nan,
            "cohorte_confiable":   confiable,
            "fallas_obs_cohorte":  fallas_obs,
            "meses_expuesto":      meses_exp,
        })

    return pd.DataFrame(rows)


cohorte_df = calcular_cohorte_ratio(df_work, hazard_curves)
print(f"✅ Cohortes analizadas: {len(cohorte_df):,}  (confiables: {cohorte_df['cohorte_confiable'].sum()})")

# Ratio consolidado por serie (máximo ratio entre sus cohortes confiables)
cohorte_serie = (
    cohorte_df[cohorte_df["cohorte_confiable"]]
    .groupby("serie_id")["cohorte_ratio"]
    .agg(cohorte_ratio_max="max", cohorte_ratio_mean="mean")
    .reset_index()
)


✅ Curvas de hazard calculadas para familias: ['211evo2', '888G3', '888evo4', '888evo5']


✅ Cohortes analizadas: 1,681  (confiables: 1681)


In [7]:
# ── FEATURE ENGINEERING (sin data leakage temporal) ──────────────────────────
# REGLA: todas las features deben tener shift ≥ 1 mes respecto al target.
# Los encoders se fittean SÓLO sobre datos de entrenamiento (ver celda de WF).

def build_features(panel: pd.DataFrame, cohorte_serie: pd.DataFrame,
                   costos_ref: pd.DataFrame, fit_encoders: bool = True,
                   encoders: dict = None) -> tuple:
    """
    Construye la matriz de features para el modelo global.
    Retorna (X_df, y, encoders_dict).

    fit_encoders=True  → fit nuevos encoders (modo entrenamiento inicial)
    fit_encoders=False → usar encoders existentes (modo incremental / validación)
    """
    df = panel.copy()
    if "serie_id" not in df.columns:
        raise ValueError("panel debe contener la columna 'serie_id'")
    df = df.sort_values(["serie_id", "mes_repair"]).reset_index(drop=True)

    # Merge metadatos de costo y cohorte
    # Eliminar columnas ya presentes para evitar duplicados en llamadas recursivas
    _merge_cols_cost    = [c for c in costos_ref.columns if c != "serie_id"]
    _merge_cols_cohorte = ["cohorte_ratio_max", "cohorte_ratio_mean"]
    df = df.drop(columns=[c for c in _merge_cols_cost + _merge_cols_cohorte if c in df.columns])
    df = df.merge(costos_ref, on="serie_id", how="left")
    df = df.merge(cohorte_serie[["serie_id","cohorte_ratio_max","cohorte_ratio_mean"]],
                  on="serie_id", how="left")

    # ── Features de lag y rolling (shift ≥ 1 → sin leakage) ─────────────────
    # Usamos .transform() vectorizado para evitar pérdida de columnas con .apply()
    df = df.sort_values(["serie_id", "mes_repair"]).reset_index(drop=True)

    grpf = df.groupby("serie_id")["fallas"]
    grpc = df.groupby("serie_id")["costo_total"]

    df["lag_1"]      = grpf.shift(1)
    df["lag_2"]      = grpf.shift(2)
    df["lag_3"]      = grpf.shift(3)
    df["lag_6"]      = grpf.shift(6)
    df["roll3_mean"] = grpf.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.rolling(3, min_periods=1).mean())
    df["roll6_mean"] = grpf.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.rolling(6, min_periods=1).mean())
    df["roll3_cost"] = grpc.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.rolling(3, min_periods=1).mean())
    df["roll6_cost"] = grpc.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.rolling(6, min_periods=1).mean())
    df["trend_3"]    = grpf.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.rolling(3, min_periods=2).apply(slope_lineal, raw=True))
    df["trend_6"]    = grpf.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.rolling(6, min_periods=3).apply(slope_lineal, raw=True))
    df["cusum_feat"] = grpf.shift(1).groupby(df["serie_id"]).transform(
                           lambda x: x.expanding().apply(cusum_stat, raw=True))
    df["mis_lag1"]   = df.groupby("serie_id")["mis_mean"].shift(1)
    df["km_lag1"]    = df.groupby("serie_id")["km_mean"].shift(1)

    # ── Features de calendario ────────────────────────────────────────────────
    df["mes_num"]    = df["mes_dt"].dt.month
    df["anio"]       = df["mes_dt"].dt.year
    df["quarter"]    = df["mes_dt"].dt.quarter
    # Codificación cíclica del mes para capturar estacionalidad
    df["mes_sin"]    = np.sin(2 * np.pi * df["mes_num"] / 12)
    df["mes_cos"]    = np.cos(2 * np.pi * df["mes_num"] / 12)

    # ── Encoders categóricos (fit sólo en train) ──────────────────────────────
    cat_cols = ["ea_number", "componente", "damage_code", "mis_cluster"]
    if encoders is None:
        encoders = {}

    for col in cat_cols:
        enc_key = f"le_{col}"
        if fit_encoders:
            le = LabelEncoder()
            df[f"{col}_enc"] = le.fit_transform(df[col].astype(str).fillna("UNK"))
            encoders[enc_key] = le
        else:
            le = encoders.get(enc_key)
            if le is not None:
                known = set(le.classes_)
                df[col] = df[col].astype(str).fillna("UNK").apply(
                    lambda x: x if x in known else le.classes_[0]
                )
                df[f"{col}_enc"] = le.transform(df[col])
            else:
                df[f"{col}_enc"] = 0

    # ── Features de cohorte (Engine Date – señal anticipada) ─────────────────
    df["cohorte_ratio"]   = df["cohorte_ratio_max"].fillna(1.0)
    df["cohorte_ratio"].clip(0, 5, inplace=True)

    # ── Costo unitario de referencia ──────────────────────────────────────────
    df["costo_unit_ref"]  = df["costo_unit_med"].fillna(df["costo_unit_med"].median())

    # ── Columnas de features finales ─────────────────────────────────────────
    FEATURE_COLS = [
        "lag_1","lag_2","lag_3","lag_6",
        "roll3_mean","roll6_mean","roll3_cost","roll6_cost",
        "trend_3","trend_6","cusum_feat",
        "mis_lag1","km_lag1",
        "cohorte_ratio",
        "mes_sin","mes_cos","quarter","anio",
        "ea_number_enc","componente_enc","damage_code_enc","mis_cluster_enc",
        "costo_unit_ref",
    ]

    # Añadir candidatas nuevas si pasan la validación estadística
    for c in CANDIDATAS_NUEVAS_NUMERICAS + CANDIDATAS_NUEVAS_CATEGORICAS:
        enc_c = f"{c}_enc" if c in CANDIDATAS_NUEVAS_CATEGORICAS else c
        if enc_c in df.columns:
            FEATURE_COLS.append(enc_c)

    # Eliminar filas sin lag_1 (primer mes de cada serie)
    df_model = df.dropna(subset=["lag_1"]).copy()
    df_model[FEATURE_COLS] = df_model[FEATURE_COLS].fillna(0)

    X = df_model[FEATURE_COLS].astype(float)
    y = df_model["fallas"].astype(float)

    return X, y, df_model, FEATURE_COLS, encoders


# Primera pasada para conocer las columnas
X_all, y_all, panel_feat, FEATURE_COLS, encoders_global = build_features(
    monthly, cohorte_serie, costos_ref, fit_encoders=True
)

print(f"✅ Features construidas: {len(FEATURE_COLS)} variables")
print(f"   Filas disponibles para modelado: {len(X_all):,}")
print("   Features:", FEATURE_COLS)


✅ Features construidas: 23 variables
   Filas disponibles para modelado: 3,923
   Features: ['lag_1', 'lag_2', 'lag_3', 'lag_6', 'roll3_mean', 'roll6_mean', 'roll3_cost', 'roll6_cost', 'trend_3', 'trend_6', 'cusum_feat', 'mis_lag1', 'km_lag1', 'cohorte_ratio', 'mes_sin', 'mes_cos', 'quarter', 'anio', 'ea_number_enc', 'componente_enc', 'damage_code_enc', 'mis_cluster_enc', 'costo_unit_ref']


In [8]:
# ── VALIDACIÓN ESTADÍSTICA DE VARIABLES (FDR Benjamini-Hochberg) ─────────────
# Se corre automáticamente para las candidatas nuevas.
# Las variables base del modelo no se someten a veto estadístico (ya están validadas).

validacion_rows = []

def test_numerica_vs_target(df: pd.DataFrame, var: str, target: str = "fallas") -> dict:
    """ANOVA + Spearman entre variable numérica y conteo de fallas."""
    clean = df[[var, target]].dropna()
    if len(clean) < 10:
        return {"variable": var, "test": "n_insuf", "p_value": 1.0}

    # Dividir en cuartiles y hacer ANOVA
    q = pd.qcut(clean[var], 4, duplicates="drop")
    groups = [g[target].values for _, g in clean.groupby(q) if len(g) >= 3]
    if len(groups) >= 2:
        _, p_anova = f_oneway(*groups)
    else:
        p_anova = 1.0

    rho, p_sp = stats.spearmanr(clean[var], clean[target])
    p_best = min(p_anova, p_sp)
    return {"variable": var, "test": "ANOVA+Spearman", "p_value": float(p_best), "rho": float(rho)}


def test_categorica_vs_target(df: pd.DataFrame, var: str, target: str = "fallas") -> dict:
    """Kruskal-Wallis entre variable categórica y conteo de fallas."""
    clean = df[[var, target]].dropna()
    if len(clean) < 10:
        return {"variable": var, "test": "n_insuf", "p_value": 1.0}
    groups = [g[target].values for _, g in clean.groupby(var) if len(g) >= 3]
    if len(groups) >= 2:
        _, p_kw = kruskal(*groups)
    else:
        p_kw = 1.0
    return {"variable": var, "test": "Kruskal-Wallis", "p_value": float(p_kw)}


# Tests para candidatas nuevas
for var in CANDIDATAS_NUEVAS_NUMERICAS:
    if var in panel_feat.columns:
        row = test_numerica_vs_target(panel_feat, var)
        validacion_rows.append(row)

for var in CANDIDATAS_NUEVAS_CATEGORICAS:
    if var in panel_feat.columns:
        row = test_categorica_vs_target(panel_feat, var)
        validacion_rows.append(row)

# Tests informativos para features base
for var in ["lag_1","roll3_mean","trend_3","cusum_feat","cohorte_ratio","mis_lag1"]:
    if var in panel_feat.columns:
        row = test_numerica_vs_target(panel_feat, var)
        validacion_rows.append(row)

if validacion_rows:
    val_df = pd.DataFrame(validacion_rows)
    p_arr  = val_df["p_value"].values
    rejected, p_adj = fdr_correction(p_arr)
    val_df["p_adj"]   = p_adj
    val_df["sig_FDR"] = rejected
    val_df = val_df.sort_values("p_adj")

    # Veto: candidatas nuevas que no pasan FDR se excluyen del modelo
    vetadas = val_df[~val_df["sig_FDR"] & val_df["variable"].isin(
        CANDIDATAS_NUEVAS_NUMERICAS + CANDIDATAS_NUEVAS_CATEGORICAS
    )]["variable"].tolist()
    if vetadas:
        print(f"\n⚠️  Variables candidatas VETADAS por FDR: {vetadas}")

    val_df.to_csv(OUTPUT_DIR / "validacion_variables.csv", index=False)
    print("✅ Validación estadística de variables:")
    print(val_df[["variable","test","p_value","p_adj","sig_FDR"]].to_string(index=False))
else:
    val_df = pd.DataFrame()
    print("ℹ️  Sin candidatas nuevas → sin validación FDR adicional")


✅ Validación estadística de variables:
     variable           test      p_value        p_adj  sig_FDR
        lag_1 ANOVA+Spearman 0.000000e+00 0.000000e+00     True
   roll3_mean ANOVA+Spearman 0.000000e+00 0.000000e+00     True
cohorte_ratio ANOVA+Spearman 1.876191e-62 3.752382e-62     True
      trend_3 ANOVA+Spearman 1.377136e-59 2.065704e-59     True
     mis_lag1 ANOVA+Spearman 2.895119e-35 3.474143e-35     True
   cusum_feat ANOVA+Spearman 1.000000e+00 1.000000e+00    False


In [9]:
# ── OPTIMIZACIÓN DE HIPERPARÁMETROS CON OPTUNA ───────────────────────────────
# Solo se ejecuta si REOPTIMIZAR_HIPERPARAMETROS=True o no existe best_params.json

BEST_PARAMS_FILE = MODEL_DIR / "best_params.json"

def _wf_wape_4folds(monthly_raw: pd.DataFrame, params: dict,
                    feat_cols: list, encs: dict) -> float:
    """
    WAPE out-of-sample en 4 folds walk-forward (usado por Optuna).
    Usa monthly_raw (panel sin features) para reconstruir features por ventana.
    """
    meses = sorted(monthly_raw["mes_repair"].unique())
    n = len(meses)
    if n < MIN_MESES_ENTRENAMIENTO + 4:
        return 1.0

    folds_wape = []
    for fold in range(4):
        corte_idx   = MIN_MESES_ENTRENAMIENTO + fold
        if corte_idx >= n:
            break
        train_meses  = meses[:corte_idx]
        val_mes      = meses[corte_idx]
        all_meses_tv = meses[:corte_idx + 1]

        # Fit encoders en train únicamente
        data_tr_raw = monthly_raw[monthly_raw["mes_repair"].isin(train_meses)]
        _, _, _, fc, encs_fold = build_features(
            data_tr_raw, cohorte_serie, costos_ref, fit_encoders=True
        )
        # Build features sobre ventana completa (train+val) con encoders de train
        data_tv = monthly_raw[monthly_raw["mes_repair"].isin(all_meses_tv)]
        _, _, feat_tv, fc, _ = build_features(
            data_tv, cohorte_serie, costos_ref, fit_encoders=False, encoders=encs_fold
        )

        train_feat = feat_tv[feat_tv["mes_repair"].isin(train_meses)]
        val_feat   = feat_tv[feat_tv["mes_repair"] == val_mes]

        X_tr = train_feat[fc].fillna(0).astype(float)
        y_tr = train_feat["fallas"].astype(float)
        X_v  = val_feat[fc].fillna(0).astype(float)
        y_v  = val_feat["fallas"].astype(float)

        if len(X_tr) < 10 or len(X_v) == 0:
            continue

        sw = sample_weights_decay(len(y_tr))
        m = XGBRegressor(**{**XGB_PARAMS_BASE, **params})
        m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        preds = np.maximum(m.predict(X_v), 0)
        w = wape(y_v.values, preds)
        if not np.isnan(w):
            folds_wape.append(w)

    return float(np.mean(folds_wape)) if folds_wape else 1.0


def optuna_objective(trial: optuna.Trial) -> float:
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators",    200, 600),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        max_depth        = trial.suggest_int("max_depth",        3, 7),
        min_child_weight = trial.suggest_int("min_child_weight", 2, 8),
        subsample        = trial.suggest_float("subsample",      0.7, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0),
        reg_alpha        = trial.suggest_float("reg_alpha",      0.1, 5.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda",     0.5, 5.0, log=True),
    )
    return _wf_wape_4folds(monthly, params, FEATURE_COLS, encoders_global)


if REOPTIMIZAR_HIPERPARAMETROS or not BEST_PARAMS_FILE.exists():
    print(f"🔍 Iniciando Optuna ({N_OPTUNA_TRIALS} trials)…")
    study = optuna.create_study(
        direction="minimize",
        pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=2)
    )
    study.optimize(optuna_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

    best_params = study.best_params
    best_wape   = study.best_value
    print(f"✅ Optuna terminado  |  Mejor WAPE 4-fold = {best_wape:.4f}")
    print(f"   Mejores parámetros: {best_params}")

    with open(BEST_PARAMS_FILE, "w") as f:
        json.dump(best_params, f, indent=2)
else:
    with open(BEST_PARAMS_FILE) as f:
        best_params = json.load(f)
    print(f"✅ Parámetros cargados de {BEST_PARAMS_FILE}")
    print(f"   {best_params}")

# Combinar con parámetros base (best_params tiene prioridad)
XGB_PARAMS_FINAL = {**XGB_PARAMS_BASE, **best_params}
print(f"\n📋 Parámetros finales XGBoost: {XGB_PARAMS_FINAL}")


✅ Parámetros cargados de modelos_ml/best_params.json
   {'n_estimators': 434, 'learning_rate': 0.04524554672516094, 'max_depth': 7, 'min_child_weight': 4, 'subsample': 0.7173352622220918, 'colsample_bytree': 0.8372144765455174, 'reg_alpha': 0.9561838825670275, 'reg_lambda': 1.148454920200187}

📋 Parámetros finales XGBoost: {'reg_alpha': 0.9561838825670275, 'reg_lambda': 1.148454920200187, 'min_child_weight': 4, 'max_depth': 7, 'subsample': 0.7173352622220918, 'colsample_bytree': 0.8372144765455174, 'n_estimators': 434, 'learning_rate': 0.04524554672516094, 'random_state': 42, 'objective': 'reg:squarederror', 'verbosity': 0}


In [10]:
# ── BACKTESTING WALK-FORWARD ─────────────────────────────────────────────────
# Ventana de entrenamiento mínima: 12 meses
# Paso: 1 mes | Mínimo 6 folds
# WAPE reportado por fold (in-sample y OOS) para detectar overfitting.

def walk_forward_validation(monthly_raw: pd.DataFrame, feat_cols: list,
                             params: dict) -> tuple:
    """
    Walk-forward sobre el panel MENSUAL RAW.
    Para cada fold:
      1. Fit encoders sólo en train_meses
      2. Build features sobre (train+val) con esos encoders
      3. Entrenar en train, predecir en val → WAPE OOS
    """
    meses    = sorted(monthly_raw["mes_repair"].unique())
    n_meses  = len(meses)
    max_folds = n_meses - MIN_MESES_ENTRENAMIENTO

    if max_folds < WF_FOLDS_MINIMOS:
        print(f"⚠️  Solo {max_folds} folds disponibles (mínimo requerido: {WF_FOLDS_MINIMOS})")

    resultados    = []
    all_residuals = []

    for fold in range(max_folds):
        corte_idx    = MIN_MESES_ENTRENAMIENTO + fold
        train_meses  = meses[:corte_idx]
        val_mes      = meses[corte_idx]
        all_meses_tv = meses[:corte_idx + 1]

        # Fit encoders SÓLO en datos de entrenamiento
        data_tr_only = monthly_raw[monthly_raw["mes_repair"].isin(train_meses)]
        _, _, _, fc, encs_fold = build_features(
            data_tr_only, cohorte_serie, costos_ref, fit_encoders=True
        )

        # Build features en ventana completa (train+val) con encoders de train
        data_tv = monthly_raw[monthly_raw["mes_repair"].isin(all_meses_tv)]
        _, _, feat_tv, fc, _ = build_features(
            data_tv, cohorte_serie, costos_ref, fit_encoders=False, encoders=encs_fold
        )

        train_feat = feat_tv[feat_tv["mes_repair"].isin(train_meses)]
        val_feat   = feat_tv[feat_tv["mes_repair"] == val_mes]

        X_tr = train_feat[fc].fillna(0).astype(float)
        y_tr = train_feat["fallas"].astype(float)
        X_v  = val_feat[fc].fillna(0).astype(float)
        y_v  = val_feat["fallas"].astype(float)

        if len(X_tr) < 10 or len(X_v) == 0 or y_v.sum() == 0:
            continue

        sw = sample_weights_decay(len(y_tr))

        # Early stopping con últimas 3 meses de train como eval set
        n_series   = monthly_raw["serie_id"].nunique()
        eval_rows  = min(3 * n_series, max(len(X_tr) // 6, 5))
        X_eval, y_eval = X_tr.iloc[-eval_rows:], y_tr.iloc[-eval_rows:]
        X_fit,  y_fit  = X_tr.iloc[:-eval_rows], y_tr.iloc[:-eval_rows]
        sw_fit = sw[:len(y_fit)]

        if len(X_fit) < 5:
            X_fit, y_fit, sw_fit = X_tr, y_tr, sw
            X_eval, y_eval = X_tr.iloc[-2:], y_tr.iloc[-2:]

        model = XGBRegressor(**{**params, "early_stopping_rounds": 30, "eval_metric": "mae"})
        model.fit(X_fit, y_fit, sample_weight=sw_fit,
                  eval_set=[(X_eval, y_eval)], verbose=False)

        preds_tr  = np.maximum(model.predict(X_tr), 0)
        wape_is   = wape(y_tr.values, preds_tr)
        preds_oos = np.maximum(model.predict(X_v),  0)
        wape_oos  = wape(y_v.values, preds_oos)
        bias_oos  = float(np.mean(preds_oos - y_v.values))
        mae_oos   = float(np.mean(np.abs(preds_oos - y_v.values)))

        gap = (wape_is - wape_oos) if not (np.isnan(wape_is) or np.isnan(wape_oos)) else np.nan
        all_residuals.extend((y_v.values - preds_oos).tolist())

        resultados.append({
            "fold": fold + 1, "mes_val": str(val_mes),
            "n_train": len(X_tr), "n_val": len(X_v),
            "wape_is": wape_is, "wape_oos": wape_oos,
            "gap_of": gap, "bias_oos": bias_oos, "mae_oos": mae_oos,
            "n_est_eff": model.best_iteration if model.best_iteration else params.get("n_estimators", 300),
        })

        if (fold + 1) % 3 == 0:
            print(f"  Fold {fold+1:2d} | {val_mes} | WAPE OOS={wape_oos:.4f} IS={wape_is:.4f} gap={gap:+.4f}")

    return pd.DataFrame(resultados), all_residuals


print("Ejecutando walk-forward backtesting…")
wf_results, wf_residuals = walk_forward_validation(monthly, FEATURE_COLS, XGB_PARAMS_FINAL)

print(f"\n{'='*60}")
print("RESULTADOS WALK-FORWARD")
print(f"{'='*60}")
print(wf_results[["fold","mes_val","n_val","wape_oos","wape_is","gap_of","bias_oos"]].to_string(index=False))

wape_oos_mean  = wf_results["wape_oos"].mean()
wape_is_mean   = wf_results["wape_is"].mean()
gap_mean       = wf_results["gap_of"].mean()
n_folds_ok     = (wf_results["wape_oos"] <= WAPE_MAX_OOS).sum()
n_folds_total  = len(wf_results)

print(f"\n  WAPE OOS promedio : {wape_oos_mean:.4f}  (límite {WAPE_MAX_OOS})")
print(f"  WAPE IS  promedio : {wape_is_mean:.4f}")
print(f"  Gap promedio       : {gap_mean:+.4f}  (límite {GAP_OVERFITTING_MAX})")
print(f"  Folds WAPE ≤ 11%  : {n_folds_ok}/{n_folds_total}")

# Calcular cobertura CI 95%
if wf_residuals:
    sigma_wf = np.std(wf_residuals)
    # Cobertura: qué % de residuos caben dentro de ±1.96σ
    within = np.mean(np.abs(wf_residuals) <= 1.96 * sigma_wf)
    print(f"  Cobertura CI 95%  : {within:.3f}  (mínimo {COBERTURA_CI_MIN})")
else:
    within = 0.0

# Guardar métricas de backtesting
wf_results.to_csv(OUTPUT_DIR / "historial_accuracy.csv", index=False)


Ejecutando walk-forward backtesting…


  Fold  3 | 2025-03 | WAPE OOS=0.4709 IS=0.3816 gap=-0.0892


  Fold  6 | 2025-06 | WAPE OOS=0.4512 IS=0.3510 gap=-0.1003


  Fold  9 | 2025-09 | WAPE OOS=0.3957 IS=0.3531 gap=-0.0426



RESULTADOS WALK-FORWARD
 fold mes_val  n_val  wape_oos  wape_is    gap_of  bias_oos
    1 2025-01    178  1.595789 0.437386 -1.158403  7.439275
    2 2025-02    195  0.797818 0.400209 -0.397609  3.031917
    3 2025-03    226  0.470854 0.381633 -0.089222 -2.226095
    4 2025-04    241  0.442659 0.351531 -0.091128 -0.472065
    5 2025-05    251  0.529166 0.350530 -0.178636  0.615582
    6 2025-06    248  0.451247 0.350962 -0.100285 -1.007143
    7 2025-07    286  0.423136 0.361619 -0.061517 -1.064349
    8 2025-08    303  0.407148 0.354666 -0.052482  0.305689
    9 2025-09    314  0.395702 0.353089 -0.042613 -0.864305
   10 2025-10    316  0.477590 0.343601 -0.133988 -2.220403
   11 2025-11    313  0.496850 0.314070 -0.182779 -1.459141

  WAPE OOS promedio : 0.5898  (límite 0.11)
  WAPE IS  promedio : 0.3636
  Gap promedio       : -0.2262  (límite 0.15)
  Folds WAPE ≤ 11%  : 0/11
  Cobertura CI 95%  : 0.985  (mínimo 0.88)


In [11]:
# ── MODELO GLOBAL XGBOOST ────────────────────────────────────────────────────
# Entrena sobre TODO el panel histórico con los hiperparámetros finales.

print("Entrenando modelo global XGBoost…")

# Re-build features sobre dataset completo con encoders globales
X_all, y_all, panel_feat_full, FEATURE_COLS, encoders_global = build_features(
    monthly, cohorte_serie, costos_ref, fit_encoders=True
)

sw_all = sample_weights_decay(len(y_all))

# Eval set: últimos 3 meses de datos
meses_all = sorted(panel_feat_full["mes_repair"].unique())
eval_meses = meses_all[-3:]
mask_eval  = panel_feat_full["mes_repair"].isin(eval_meses)

X_eval_g  = X_all[mask_eval].values
y_eval_g  = y_all[mask_eval].values
X_fit_g   = X_all[~mask_eval].values
y_fit_g   = y_all[~mask_eval].values
sw_fit    = sample_weights_decay(len(y_fit_g))

model_global = XGBRegressor(**{**XGB_PARAMS_FINAL, "early_stopping_rounds": 30, "eval_metric": "mae"})
model_global.fit(
    X_fit_g, y_fit_g,
    sample_weight = sw_fit,
    eval_set      = [(X_eval_g, y_eval_g)],
    verbose       = False
)

n_est_eff_global = model_global.best_iteration if model_global.best_iteration else XGB_PARAMS_FINAL["n_estimators"]
preds_tr_global  = np.maximum(model_global.predict(X_all.values), 0)
wape_is_global   = wape(y_all.values, preds_tr_global)

print(f"✅ Modelo global entrenado")
print(f"   n_estimators efectivos : {n_est_eff_global}")
print(f"   WAPE in-sample          : {wape_is_global:.4f}")

# Importancia de features
feat_imp = pd.Series(model_global.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(f"\n   Top-10 features por importancia:")
print(feat_imp.head(10).to_string())

# Guardar modelo global
joblib.dump(model_global, MODEL_DIR / "global_xgb.pkl")
joblib.dump(encoders_global, MODEL_DIR / "encoders.pkl")
print(f"\n💾 Modelo global guardado en {MODEL_DIR}/global_xgb.pkl")


Entrenando modelo global XGBoost…


✅ Modelo global entrenado
   n_estimators efectivos : 232
   WAPE in-sample          : 0.2969

   Top-10 features por importancia:
lag_1              0.594332
roll3_mean         0.228473
trend_6            0.037422
roll6_mean         0.020486
ea_number_enc      0.016086
mis_cluster_enc    0.015200
lag_2              0.010557
trend_3            0.009508
lag_3              0.007756
anio               0.007140

💾 Modelo global guardado en modelos_ml/global_xgb.pkl


In [12]:
# ── MODELOS INDIVIDUALES POR SERIE ───────────────────────────────────────────
# Series con ≥ 12 meses → early stopping | < 12 meses → n_estimators=100 fijo

def entrenar_modelo_individual(serie_id: str, g: pd.DataFrame,
                                costos_ref: pd.DataFrame,
                                cohorte_serie: pd.DataFrame) -> dict:
    """
    Entrena XGBoost individual para una serie.
    Retorna dict con modelo, n_est_efectivo, wape_is, wape_oos_local.
    """
    g = g.sort_values("mes_repair").copy()
    n_meses = len(g)

    if n_meses < MIN_MESES_SERIE:
        return None

    # Features individuales (encoders propios por serie)
    panel_s = g.copy()
    _, _, feat_s, fc_s, enc_s = build_features(
        panel_s, cohorte_serie, costos_ref, fit_encoders=True
    )
    X_s = feat_s[fc_s].fillna(0).astype(float).values
    y_s = feat_s["fallas"].astype(float).values

    if len(X_s) < 4:
        return None

    sw_s = sample_weights_decay(len(y_s))

    if n_meses >= MIN_MESES_ENTRENAMIENTO:
        # Early stopping con últimos 3 puntos como eval
        X_ev, y_ev = X_s[-3:], y_s[-3:]
        X_fi, y_fi = X_s[:-3], y_s[:-3]
        if len(X_fi) < 4:
            X_fi, y_fi = X_s, y_s
            X_ev, y_ev = X_s[-2:], y_s[-2:]
        sw_fi = sample_weights_decay(len(y_fi))
        m = XGBRegressor(**{**XGB_PARAMS_FINAL, "early_stopping_rounds": 30, "eval_metric": "mae"})
        m.fit(X_fi, y_fi, sample_weight=sw_fi, eval_set=[(X_ev, y_ev)], verbose=False)
        n_eff = m.best_iteration if m.best_iteration else XGB_PARAMS_FINAL["n_estimators"]
    else:
        # n_estimators fijo para series cortas
        params_short = {**XGB_PARAMS_FINAL, "n_estimators": 100}
        params_short.pop("early_stopping_rounds", None)
        m = XGBRegressor(**params_short)
        m.fit(X_s, y_s, sample_weight=sw_s, verbose=False)
        n_eff = 100

    preds_s   = np.maximum(m.predict(X_s), 0)
    wape_s    = wape(y_s, preds_s)

    return {
        "model":   m,
        "n_eff":   n_eff,
        "wape_is": wape_s,
        "n_meses": n_meses,
        "encoders": enc_s,
        "fc":       fc_s,
    }


# Entrenar modelos individuales para series con datos suficientes
modelos_individuales = {}
series_validas = panel_feat_full.groupby("serie_id").filter(
    lambda g: len(g) >= MIN_MESES_SERIE
)["serie_id"].unique()

n_ind = 0
wape_ind_list = []
for sid in series_validas:
    g_s = monthly[monthly["serie_id"] == sid]
    res = entrenar_modelo_individual(sid, g_s, costos_ref, cohorte_serie)
    if res is not None:
        modelos_individuales[sid] = res
        joblib.dump(res, MODEL_DIR / "individuales" / f"{hash_serie(sid)}.pkl")
        wape_ind_list.append(res["wape_is"])
        n_ind += 1

print(f"✅ Modelos individuales entrenados: {n_ind}")
if wape_ind_list:
    print(f"   WAPE IS medio (individuales): {np.mean(wape_ind_list):.4f}")
    print(f"   n_estimators efectivo medio : {np.mean([r['n_eff'] for r in modelos_individuales.values()]):.0f}")


✅ Modelos individuales entrenados: 247
   WAPE IS medio (individuales): 0.4197
   n_estimators efectivo medio : 128


In [13]:
# ── FORECAST RECURSIVO 6 MESES ───────────────────────────────────────────────
# Usa modelo individual si existe, sino modelo global.
# El cap_forecast se aplica en cada paso con historia_real (nunca predicciones).

def forecast_serie(serie_id: str, monthly_panel: pd.DataFrame,
                   model_global: object, modelos_ind: dict,
                   feat_cols: list, encoders: dict,
                   costos_ref: pd.DataFrame, cohorte_serie: pd.DataFrame,
                   wf_residuals: list,
                   n_meses: int = HORIZONTE_MESES) -> pd.DataFrame:
    """Genera forecast recursivo para una serie. Retorna DataFrame de predicciones."""
    g = monthly_panel[monthly_panel["serie_id"] == serie_id].sort_values("mes_repair").copy()
    if len(g) == 0:
        return pd.DataFrame()

    historia_real = g["fallas"].values.copy()   # solo valores reales
    ultimo_mes    = g["mes_repair"].iloc[-1]
    ultimo_dt     = g["mes_dt"].iloc[-1]

    # Metadatos estáticos de la serie
    ea       = g["ea_number"].iloc[-1]
    comp     = g["componente"].iloc[-1]
    dc       = g["damage_code"].iloc[-1]
    mis_cl   = g["mis_cluster"].iloc[-1]

    # Contexto de costo
    costo_ref_row = costos_ref[costos_ref["serie_id"] == serie_id]
    costo_unit    = float(costo_ref_row["costo_unit_med"].values[0]) if len(costo_ref_row) else 0.0

    # Seleccionar modelo
    usar_individual = serie_id in modelos_ind
    if usar_individual:
        m_ind      = modelos_ind[serie_id]["model"]
        enc_ind    = modelos_ind[serie_id]["encoders"]
        fc_ind     = modelos_ind[serie_id]["fc"]
    residuals_arr = np.array(wf_residuals) if wf_residuals else np.array([0.0])

    predicciones = []
    # Estado deslizante para el forecast recursivo (no se mezcla con historia_real)
    ventana      = g.copy()

    for h in range(n_meses):
        sig_mes_period = ultimo_mes + (h + 1)
        sig_mes_dt     = sig_mes_period.to_timestamp()

        # Construir fila de features para el siguiente mes
        _, _, feat_next, fc_g, _ = build_features(
            ventana, cohorte_serie, costos_ref, fit_encoders=False, encoders=encoders
        )
        if len(feat_next) == 0:
            break

        row_feat = feat_next[fc_g].iloc[[-1]].fillna(0).astype(float)

        if usar_individual:
            row_ind = feat_next[fc_ind].iloc[[-1]].fillna(0).astype(float) if fc_ind == fc_g else row_feat[fc_ind] if all(c in row_feat.columns for c in fc_ind) else row_feat
            pred_i  = float(np.maximum(m_ind.predict(row_ind.values), 0)[0])
            pred_g  = float(np.maximum(model_global.predict(row_feat.values), 0)[0])
            # Blend: 60% individual, 40% global
            pred_raw = 0.6 * pred_i + 0.4 * pred_g
        else:
            pred_raw = float(np.maximum(model_global.predict(row_feat.values), 0)[0])

        # Cap conservador: usa SÓLO historia_real
        pred_cap = cap_forecast(pred_raw, historia_real)

        lo, hi = ci_bandas(np.array([pred_cap] * (h + 1)), residuals_arr)
        lo_h, hi_h = float(lo[-1]), float(hi[-1])

        predicciones.append({
            "serie_id":      serie_id,
            "ea_number":     ea,
            "componente":    comp,
            "damage_code":   dc,
            "mis_cluster":   mis_cl,
            "mes_forecast":  str(sig_mes_period),
            "mes_dt":        sig_mes_dt,
            "horizonte":     h + 1,
            "fallas_central": pred_cap,
            "fallas_lo":     lo_h,
            "fallas_hi":     hi_h,
            "costo_central": pred_cap * costo_unit,
            "costo_lo":      lo_h  * costo_unit,
            "costo_hi":      hi_h  * costo_unit,
            "modelo_usado":  "blend" if usar_individual else "global",
        })

        # Agregar fila sintética a la ventana para el siguiente paso recursivo
        nueva_fila = ventana.iloc[[-1]].copy()
        nueva_fila["mes_repair"] = sig_mes_period
        nueva_fila["mes_dt"]     = sig_mes_dt
        nueva_fila["fallas"]     = pred_cap          # se usa para lag en h+2
        nueva_fila["costo_total"]= pred_cap * costo_unit
        ventana = pd.concat([ventana, nueva_fila], ignore_index=True)
        # NOTA: historia_real permanece inalterada (solo datos reales)

    return pd.DataFrame(predicciones)


# Generar forecasts para TODAS las series con suficientes datos
print("Generando forecast 6 meses para todas las series…")
forecast_rows = []

series_forecast = monthly["serie_id"].unique()
for i, sid in enumerate(series_forecast):
    g_hist = monthly[monthly["serie_id"] == sid]
    if g_hist["fallas"].sum() == 0 or len(g_hist) < 2:
        continue
    fc_df = forecast_serie(
        sid, monthly, model_global, modelos_individuales,
        FEATURE_COLS, encoders_global, costos_ref, cohorte_serie, wf_residuals
    )
    forecast_rows.append(fc_df)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(series_forecast)} series procesadas…")

forecast_df = pd.concat(forecast_rows, ignore_index=True) if forecast_rows else pd.DataFrame()
print(f"\n✅ Forecast generado: {len(forecast_df):,} filas ({forecast_df['serie_id'].nunique()} series × 6 meses)")

# Verificar cap: forecast mes 6 ≤ 1.5× máximo histórico real por serie
violaciones_cap = 0
for sid, g_fc in forecast_df[forecast_df["horizonte"] == 6].groupby("serie_id"):
    hist_max = monthly[monthly["serie_id"] == sid]["fallas"].max()
    fc6 = g_fc["fallas_central"].iloc[0]
    if fc6 > hist_max * 1.5 + 0.01:
        violaciones_cap += 1
print(f"   Violaciones de cap (fc_M6 > 1.5× hist_max): {violaciones_cap}")

# Proyección de costo total a 6 meses por serie
costo_proy_6m = (forecast_df.groupby("serie_id")["costo_central"].sum()
                 .rename("costo_proj_6m").reset_index())
fallas_proy_6m = (forecast_df.groupby("serie_id")["fallas_central"].sum()
                  .rename("fallas_proj_6m").reset_index())


Generando forecast 6 meses para todas las series…


  50/890 series procesadas…


  100/890 series procesadas…


  150/890 series procesadas…


  250/890 series procesadas…


...

  750/890 series procesadas…


  800/890 series procesadas…


  850/890 series procesadas…



✅ Forecast generado: 3,510 filas (585 series × 6 meses)


   Violaciones de cap (fc_M6 > 1.5× hist_max): 0


In [14]:
# ── ISOLATION FOREST (ANOMALÍAS MULTIVARIADAS) ───────────────────────────────
# Entrena sobre la matriz de features del histórico completo.
# Detecta combinaciones inusuales de lag/rolling/cohorte.

IF_FEATURES = [
    "lag_1","lag_2","lag_3","lag_6",
    "roll3_mean","roll6_mean","roll3_cost","roll6_cost",
    "trend_3","trend_6","cusum_feat",
    "mis_lag1","km_lag1","cohorte_ratio",
]
# Solo columnas que existen en panel
IF_FEATURES = [c for c in IF_FEATURES if c in panel_feat_full.columns]

X_if = panel_feat_full[IF_FEATURES].fillna(0).astype(float)

scaler_if = StandardScaler()
X_if_sc   = scaler_if.fit_transform(X_if)

iso_forest = IsolationForest(
    n_estimators = 200,
    contamination = 0.05,
    random_state  = 42,
    n_jobs        = -1,
)
iso_forest.fit(X_if_sc)

# Score de anomalía: más negativo = más anómalo → invertir y normalizar 0-1
raw_scores = iso_forest.score_samples(X_if_sc)
# Escalar a [0, 1] donde 1 = más anómalo
if_score_norm = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-10)

panel_feat_full["if_score"] = if_score_norm

# Agregar IF score por serie (máximo de sus observaciones recientes)
IF_score_serie = (
    panel_feat_full.sort_values("mes_repair")
    .groupby("serie_id")
    .apply(lambda g: g["if_score"].tail(3).mean())
    .rename("if_score_reciente")
    .reset_index()
)

# Guardar modelo IF
joblib.dump({"model": iso_forest, "scaler": scaler_if, "features": IF_FEATURES},
            MODEL_DIR / "isolation_forest.pkl")

n_anomalos = (if_score_norm > 0.7).sum()
print(f"✅ Isolation Forest entrenado  |  {n_anomalos:,} observaciones con score > 0.7")
print(f"   Features usadas: {IF_FEATURES}")


✅ Isolation Forest entrenado  |  81 observaciones con score > 0.7
   Features usadas: ['lag_1', 'lag_2', 'lag_3', 'lag_6', 'roll3_mean', 'roll6_mean', 'roll3_cost', 'roll6_cost', 'trend_3', 'trend_6', 'cusum_feat', 'mis_lag1', 'km_lag1', 'cohorte_ratio']


In [15]:
# ── CUSUM Y DETECCIÓN ESTADÍSTICA ───────────────────────────────────────────
# Calcula por serie:
#   1. cusum_final       – CUSUM acumulado positivo normalizado
#   2. z_score_reciente  – desviación estadística del período reciente vs histórico
#   3. aceleracion_3m    – pendiente de los últimos 3 meses

def detectar_anomalias_serie(g: pd.DataFrame) -> dict:
    """Calcula métricas de anomalía para una serie mensual."""
    g = g.sort_values("mes_repair").copy()
    fallas_arr = g["fallas"].values.astype(float)
    n = len(fallas_arr)

    # CUSUM
    cs = cusum_stat(fallas_arr) if n >= 3 else 0.0

    # Z-score: media últimos 3 meses vs media histórica completa
    if n >= 6:
        mu_hist   = float(np.mean(fallas_arr[:-3]))
        sigma_hist= float(np.std(fallas_arr[:-3])) + 1e-8
        mu_rec    = float(np.mean(fallas_arr[-3:]))
        z         = (mu_rec - mu_hist) / sigma_hist
    else:
        z = 0.0

    # Aceleración (pendiente últimos 3 meses)
    accel = slope_lineal(fallas_arr[-3:]) if n >= 3 else 0.0

    # Mann-Whitney: últimos 3 meses vs histórico anterior (solo si n ≥ 8)
    if n >= 8:
        try:
            _, p_mw = mannwhitneyu(fallas_arr[-3:], fallas_arr[:-3], alternative="greater")
        except Exception:
            p_mw = 1.0
    else:
        p_mw = 1.0

    return {
        "cusum_final":       cs,
        "z_score_reciente":  z,
        "aceleracion_3m":    accel,
        "p_mw":              p_mw,
        "fallas_mean_hist":  float(np.mean(fallas_arr)),
        "fallas_max_hist":   float(np.max(fallas_arr)),
        "fallas_recientes":  float(np.mean(fallas_arr[-3:])) if n >= 3 else float(fallas_arr[-1]),
    }


anomalia_rows = []
for sid, g_s in monthly.groupby("serie_id"):
    res = detectar_anomalias_serie(g_s)
    res["serie_id"] = sid
    anomalia_rows.append(res)

anomalia_df = pd.DataFrame(anomalia_rows)

# Corrección FDR sobre los p-valores de Mann-Whitney
if "p_mw" in anomalia_df.columns:
    rejected_mw, p_adj_mw = fdr_correction(anomalia_df["p_mw"].values)
    anomalia_df["p_mw_adj"] = p_adj_mw
    anomalia_df["sig_mw"]   = rejected_mw
else:
    anomalia_df["sig_mw"] = False

print(f"✅ Detección estadística completada para {len(anomalia_df)} series")
print(f"   Series con CUSUM > 2   : {(anomalia_df['cusum_final'] > 2).sum()}")
print(f"   Series con Z-score > 2 : {(anomalia_df['z_score_reciente'] > 2).sum()}")
print(f"   Series sig. Mann-Whitney (FDR): {anomalia_df['sig_mw'].sum()}")


✅ Detección estadística completada para 890 series
   Series con CUSUM > 2   : 206
   Series con Z-score > 2 : 73
   Series sig. Mann-Whitney (FDR): 0


In [16]:
# ── SCORE DE ALERTA 7 COMPONENTES ───────────────────────────────────────────
# Cada componente se normaliza a [0,1] antes de ponderar.
# El score final es 0-100 (mayor = más urgente para el analista).

def normalizar_col(s: pd.Series) -> pd.Series:
    """Min-max normalización robusta con percentil 99 como máximo."""
    lo = s.min()
    hi = s.quantile(0.99)
    if hi <= lo:
        return pd.Series(0.0, index=s.index)
    return ((s.clip(lo, hi) - lo) / (hi - lo)).clip(0, 1)


def calcular_score(
    anomalia_df:    pd.DataFrame,
    IF_score:       pd.DataFrame,
    forecast_df:    pd.DataFrame,
    monthly:        pd.DataFrame,
    costos_ref:     pd.DataFrame,
    cohorte_serie:  pd.DataFrame,
    pesos:          dict,
    prev_forecast:  pd.DataFrame = None,
) -> pd.DataFrame:
    """Ensambla el score de 7 componentes por serie_id."""

    # ── Base: anomalía estadística (Repair Date) ─────────────────────────────
    base = anomalia_df[["serie_id","cusum_final","z_score_reciente",
                         "aceleracion_3m","fallas_mean_hist","fallas_max_hist",
                         "fallas_recientes","p_mw_adj","sig_mw"]].copy()

    # ── C1: Desviación estadística (z-score) ─────────────────────────────────
    base["c1_desviacion"] = normalizar_col(base["z_score_reciente"].clip(lower=0))

    # ── C2: Aceleración CUSUM ─────────────────────────────────────────────────
    base["c2_cusum"] = normalizar_col(base["cusum_final"].clip(lower=0))

    # ── C3: Señal Engine Date (cohorte_ratio) ─────────────────────────────────
    base = base.merge(cohorte_serie[["serie_id","cohorte_ratio_max"]].fillna(1.0),
                      on="serie_id", how="left")
    base["cohorte_ratio_max"] = base["cohorte_ratio_max"].fillna(1.0)
    # Señal anticipada: ratio > 1 indica cohortes fallando más de lo esperado
    base["c3_engine_date"] = normalizar_col((base["cohorte_ratio_max"] - 1.0).clip(lower=0))

    # ── C4: Anomalía Isolation Forest ────────────────────────────────────────
    base = base.merge(IF_score_serie, on="serie_id", how="left")
    base["if_score_reciente"] = base["if_score_reciente"].fillna(0)
    base["c4_if"] = normalizar_col(base["if_score_reciente"])

    # ── C5: Error forecast previo ────────────────────────────────────────────
    if prev_forecast is not None and len(prev_forecast) > 0:
        # Comparar forecast anterior vs realidad reciente
        ult_mes_hist = monthly.groupby("serie_id")["fallas"].last().rename("fallas_real")
        prev_f       = prev_forecast.groupby("serie_id")["fallas_central"].first().rename("fallas_pred_prev")
        err_df       = ult_mes_hist.to_frame().join(prev_f)
        err_df["error_rel"] = (np.abs(err_df["fallas_real"] - err_df["fallas_pred_prev"]) /
                               (err_df["fallas_real"].abs() + 1e-8))
        base = base.merge(err_df["error_rel"].reset_index(), on="serie_id", how="left")
        base["c5_error_prev"] = normalizar_col(base["error_rel"].fillna(0))
    else:
        base["c5_error_prev"] = 0.0

    # ── C6: Crecimiento proyectado ────────────────────────────────────────────
    # Promedio 6 meses proyectados vs promedio últimos 6 históricos
    hist_tail6 = (monthly.groupby("serie_id")
                  .apply(lambda g: g.sort_values("mes_repair")["fallas"].tail(6).mean())
                  .rename("mean_hist6").reset_index())
    fc_mean6   = (forecast_df.groupby("serie_id")["fallas_central"].mean()
                  .rename("mean_fc6").reset_index())
    growth_df  = hist_tail6.merge(fc_mean6, on="serie_id", how="left")
    growth_df["growth_ratio"] = (growth_df["mean_fc6"] /
                                 (growth_df["mean_hist6"].abs() + 1e-8))
    base = base.merge(growth_df[["serie_id","growth_ratio"]], on="serie_id", how="left")
    base["c6_crecimiento"] = normalizar_col((base["growth_ratio"].fillna(1) - 1).clip(lower=0))

    # ── C7: Costo relativo ───────────────────────────────────────────────────
    costo_total_serie = (monthly.groupby("serie_id")["costo_total"].sum()
                         .rename("costo_hist_total").reset_index())
    base = base.merge(costo_total_serie, on="serie_id", how="left")
    base["c7_costo"] = normalizar_col(base["costo_hist_total"].fillna(0))

    # ── Score ponderado ──────────────────────────────────────────────────────
    base["score"] = (
        pesos["desviacion_stat"]     * base["c1_desviacion"] +
        pesos["aceleracion_cusum"]   * base["c2_cusum"] +
        pesos["senal_engine_date"]   * base["c3_engine_date"] +
        pesos["anomalia_if"]         * base["c4_if"] +
        pesos["error_forecast_prev"] * base["c5_error_prev"] +
        pesos["crecimiento_proy"]    * base["c6_crecimiento"] +
        pesos["costo_relativo"]      * base["c7_costo"]
    ) * 100

    # ── Nivel de alerta ───────────────────────────────────────────────────────
    base["nivel_alerta"] = pd.cut(
        base["score"],
        bins   = [-1, 33, 66, 101],
        labels = ["BAJA", "MEDIA", "ALTA"]
    )

    # ── Razones explicativas (texto para el analista) ─────────────────────────
    def razones(row):
        r = []
        if row["c1_desviacion"] > 0.5:   r.append(f"Desv.stat={row['z_score_reciente']:.1f}σ")
        if row["c2_cusum"]      > 0.5:   r.append(f"CUSUM={row['cusum_final']:.1f}")
        if row["c3_engine_date"]> 0.5:   r.append(f"Cohorte={row['cohorte_ratio_max']:.2f}x")
        if row["c4_if"]         > 0.5:   r.append("IF=anomalía")
        if row.get("c5_error_prev", 0)>0.5: r.append("ErrFcPrev>50%")
        if row["c6_crecimiento"] > 0.5:  r.append(f"Crec={row.get('growth_ratio',1):.1f}x")
        if row["c7_costo"]      > 0.5:   r.append("CostoAlto")
        return " | ".join(r) if r else "Normal"

    base["razones"] = base.apply(razones, axis=1)

    return base.sort_values("score", ascending=False).reset_index(drop=True)


# Cargar forecast previo si existe (para c5)
prev_fc_file = MODEL_DIR / "forecast_historico.json"
prev_forecast_df = None
if prev_fc_file.exists():
    try:
        prev_forecast_df = pd.read_json(prev_fc_file)
    except Exception:
        prev_forecast_df = None

score_df = calcular_score(
    anomalia_df, IF_score_serie, forecast_df,
    monthly, costos_ref, cohorte_serie,
    PESOS_SCORE, prev_forecast_df
)

print(f"✅ Score calculado para {len(score_df)} series")
print(f"   ALTA  : {(score_df['nivel_alerta']=='ALTA').sum()}")
print(f"   MEDIA : {(score_df['nivel_alerta']=='MEDIA').sum()}")
print(f"   BAJA  : {(score_df['nivel_alerta']=='BAJA').sum()}")
print(f"\n   Top 5 series por score:")
print(score_df[["serie_id","score","nivel_alerta","razones"]].head(5).to_string(index=False))


✅ Score calculado para 890 series
   ALTA  : 0
   MEDIA : 23
   BAJA  : 867

   Top 5 series por score:
                                                          serie_id     score nivel_alerta                                                                razones
888evo5||Coolant pump||MECHANICAL FAULT,CRACKED, DEFORMED, DAMAGED 59.051971        MEDIA Desv.stat=133333333.3σ | CUSUM=399999998.5 | ErrFcPrev>50% | Crec=2.1x
        888evo5||Intake manifold pressure sender||ELECTRICAL FAULT 51.245717        MEDIA                 Desv.stat=266666666.7σ | CUSUM=799999998.5 | Crec=2.4x
                          888evo5||Ignition coil||ELECTRICAL FAULT 50.675199        MEDIA                Desv.stat=466666666.7σ | CUSUM=1499999999.0 | Crec=2.0x
888G3||Cylinder block||MECHANICAL FAULT,CRACKED, DEFORMED, DAMAGED 47.594503        MEDIA                Cohorte=1.00x | IF=anomalía | ErrFcPrev>50% | CostoAlto
                                  211evo2||Cylinder block||LEAKING 46.380237        MEDIA  

In [17]:
# ── WATCHLIST TOP 50 ─────────────────────────────────────────────────────────

# Combinar score con proyecciones y datos históricos
n_hist_serie = monthly.groupby("serie_id").agg(
    fallas_historicas = ("fallas",      "sum"),
    costo_historico   = ("costo_total", "sum"),
    primera_falla     = ("mes_dt",      "min"),
    ultima_falla      = ("mes_dt",      "max"),
).reset_index()

watchlist = (
    score_df
    .merge(costo_proy_6m,   on="serie_id", how="left")
    .merge(fallas_proy_6m,  on="serie_id", how="left")
    .merge(n_hist_serie,    on="serie_id", how="left")
    .merge(cohorte_serie[["serie_id","cohorte_ratio_max","cohorte_ratio_mean"]],
           on="serie_id", how="left")
)

watchlist["costo_proj_6m"]  = watchlist["costo_proj_6m"].fillna(0)
watchlist["fallas_proj_6m"] = watchlist["fallas_proj_6m"].fillna(0)

# Extraer componentes del serie_id para columnas independientes
watchlist[["ea_number_w","componente_w","damage_code_w"]] = (
    watchlist["serie_id"].str.split("||", expand=True).iloc[:, :3]
)

# Top 50 por score
top50 = watchlist.head(50).copy()

# Añadir columnas c_* para el dashboard de Power BI
for col in ["c1_desviacion","c2_cusum","c3_engine_date","c4_if",
            "c5_error_prev","c6_crecimiento","c7_costo"]:
    if col not in top50.columns:
        top50[col] = 0.0

top50 = top50.rename(columns={
    "ea_number_w": "ea_number",
    "componente_w": "componente_pw",
    "damage_code_w": "damage_code_pw",
    "cohorte_ratio_max": "cohorte_ratio",
})

print(f"✅ Watchlist Top 50 generada")
print(top50[["serie_id","score","nivel_alerta","fallas_proj_6m",
             "costo_proj_6m","razones"]].head(10).to_string(index=False))


✅ Watchlist Top 50 generada
                                                                  serie_id     score nivel_alerta  fallas_proj_6m  costo_proj_6m                                                                razones
        888evo5||Coolant pump||MECHANICAL FAULT,CRACKED, DEFORMED, DAMAGED 59.051971        MEDIA       20.895589    1544.288496 Desv.stat=133333333.3σ | CUSUM=399999998.5 | ErrFcPrev>50% | Crec=2.1x
                888evo5||Intake manifold pressure sender||ELECTRICAL FAULT 51.245717        MEDIA       33.599533    4734.342164                 Desv.stat=266666666.7σ | CUSUM=799999998.5 | Crec=2.4x
                                  888evo5||Ignition coil||ELECTRICAL FAULT 50.675199        MEDIA       51.386869    7843.177794                Desv.stat=466666666.7σ | CUSUM=1499999999.0 | Crec=2.0x
        888G3||Cylinder block||MECHANICAL FAULT,CRACKED, DEFORMED, DAMAGED 47.594503        MEDIA       83.249240  528389.170686                Cohorte=1.00x | IF=anomalía 

In [18]:
# ── EXPORTAR 8 CSVs PARA POWER BI ───────────────────────────────────────────
# Todas las rutas son relativas al directorio de trabajo (Path())

fecha_calc = datetime.now().strftime("%Y-%m-%d %H:%M")

# ── 1. watchlist_top50.csv ────────────────────────────────────────────────────
cols_top50 = [
    "serie_id","componente_pw","damage_code_pw","mis_cluster","score","nivel_alerta",
    "razones","cohorte_ratio","costo_proj_6m","fallas_proj_6m",
    "c1_desviacion","c2_cusum","c3_engine_date","c4_if",
    "c5_error_prev","c6_crecimiento","c7_costo",
]
cols_top50 = [c for c in cols_top50 if c in top50.columns]
top50[cols_top50].to_csv(OUTPUT_DIR / "watchlist_top50.csv", index=False)

# ── 2. componentes_nuevos.csv ─────────────────────────────────────────────────
# Series con historia < 6 meses → requieren revisión manual del analista
nuevos_mask = (n_hist_serie["fallas_historicas"] > 0) &               (monthly.groupby("serie_id").size().reindex(n_hist_serie["serie_id"]).fillna(0).values < MIN_MESES_SERIE)
comp_nuevos = n_hist_serie[nuevos_mask].copy()
comp_nuevos.to_csv(OUTPUT_DIR / "componentes_nuevos.csv", index=False)

# ── 3. tendencias_forecast.csv ────────────────────────────────────────────────
# Histórico + forecast ML con bandas (para gráficos de tendencia)
hist_tend = monthly[["serie_id","mes_dt","fallas","costo_total"]].copy()
hist_tend["tipo"]         = "Histórico"
hist_tend["fallas_lo"]    = hist_tend["fallas"]
hist_tend["fallas_hi"]    = hist_tend["fallas"]
hist_tend["costo_central"]= hist_tend["costo_total"]
hist_tend["costo_lo"]     = hist_tend["costo_total"]
hist_tend["costo_hi"]     = hist_tend["costo_total"]
hist_tend = hist_tend.rename(columns={"fallas": "fallas_central"})

fc_tend = forecast_df[["serie_id","mes_dt","fallas_central","fallas_lo","fallas_hi",
                        "costo_central","costo_lo","costo_hi"]].copy()
fc_tend["tipo"] = "Forecast ML"

tendencias = pd.concat([
    hist_tend[["serie_id","mes_dt","tipo","fallas_central","fallas_lo","fallas_hi",
               "costo_central","costo_lo","costo_hi"]],
    fc_tend
], ignore_index=True).sort_values(["serie_id","mes_dt"])
tendencias.to_csv(OUTPUT_DIR / "tendencias_forecast.csv", index=False)

# ── 4. heatmap_aceleracion.csv ────────────────────────────────────────────────
top20_series = top50["serie_id"].head(20).tolist()
heatmap_df = (
    monthly[monthly["serie_id"].isin(top20_series)]
    [["serie_id","mes_dt","fallas"]]
    .rename(columns={"mes_dt":"mes_repair","fallas":"fallas"})
    .copy()
)
heatmap_df["componente"] = heatmap_df["serie_id"].str.split("||").str[1]
heatmap_df["damage_code"]= heatmap_df["serie_id"].str.split("||").str[2]
heatmap_df.to_csv(OUTPUT_DIR / "heatmap_aceleracion.csv", index=False)

# ── 5. serie_mensual_completa.csv ─────────────────────────────────────────────
cols_fact = ["serie_id","mes_dt","fallas","costo_total","mis_mean","km_mean",
             "ea_number","componente","damage_code","mis_cluster"]
cols_fact = [c for c in cols_fact if c in monthly.columns]
monthly[cols_fact].to_csv(OUTPUT_DIR / "serie_mensual_completa.csv", index=False)

# ── 6. score_diagnostico.csv ─────────────────────────────────────────────────
diag_cols = ["serie_id","score","nivel_alerta","c1_desviacion","c2_cusum",
             "c3_engine_date","c4_if","c5_error_prev","c6_crecimiento","c7_costo",
             "razones","cusum_final","z_score_reciente","aceleracion_3m",
             "cohorte_ratio_max","if_score_reciente"]
diag_cols = [c for c in diag_cols if c in score_df.columns]
score_diag = score_df[diag_cols].copy()
score_diag["fecha_calculo"] = fecha_calc
score_diag.to_csv(OUTPUT_DIR / "score_diagnostico.csv", index=False)

# ── 7. validacion_variables.csv ───────────────────────────────────────────────
if len(val_df) > 0:
    val_df["fecha_calculo"] = fecha_calc
    val_df.to_csv(OUTPUT_DIR / "validacion_variables.csv", index=False)

# ── 8. historial_accuracy.csv ────────────────────────────────────────────────
acc_cols = ["fold","mes_val","n_train","n_val","wape_oos","wape_is","gap_of","bias_oos","mae_oos"]
acc_cols = [c for c in acc_cols if c in wf_results.columns]
acc_df = wf_results[acc_cols].copy()
acc_df["fecha_calculo"] = fecha_calc
acc_df["modo"] = MODO
acc_df.to_csv(OUTPUT_DIR / "historial_accuracy.csv", index=False)

# ── Guardar forecast histórico para comparar en el próximo run ───────────────
forecast_df.to_json(MODEL_DIR / "forecast_historico.json", orient="records")

# ── Guardar metadata del modelo ───────────────────────────────────────────────
metadata = {
    "version":         "v6",
    "fecha":           fecha_calc,
    "modo":            MODO,
    "n_series":        int(monthly["serie_id"].nunique()),
    "wape_oos_mean":   float(wf_results["wape_oos"].mean()) if len(wf_results) else None,
    "wape_is_mean":    float(wf_results["wape_is"].mean())  if len(wf_results) else None,
    "gap_of_mean":     float(wf_results["gap_of"].mean())   if len(wf_results) else None,
    "n_folds":         int(len(wf_results)),
    "horizonte_meses": HORIZONTE_MESES,
    "features":        FEATURE_COLS,
    "xgb_params":      XGB_PARAMS_FINAL,
}
with open(MODEL_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("✅ 8 CSVs exportados a", OUTPUT_DIR)
for fp in sorted(OUTPUT_DIR.glob("*.csv")):
    n = len(pd.read_csv(fp))
    print(f"   {fp.name:40s} {n:6,} filas")


✅ 8 CSVs exportados a outputs
   componentes_nuevos.csv                      601 filas
   heatmap_aceleracion.csv                     221 filas
   historial_accuracy.csv                       11 filas
   score_diagnostico.csv                       890 filas


   serie_mensual_completa.csv                4,813 filas
   tendencias_forecast.csv                   8,323 filas
   validacion_variables.csv                      6 filas
   watchlist_top50.csv                          50 filas


In [19]:
# ── VIZ 1: LÍNEA DE TIEMPO GLOBAL + FORECAST 6 MESES ─────────────────────────
# Eje X: Repair Date | Forecast conecta suavemente con el último histórico

def viz1_timeline_global(monthly, forecast_df, output_dir):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), facecolor=COLOR_LIGHT)

    for ax in (ax1, ax2):
        ax.set_facecolor("white")

    # Agregar totales mensuales
    hist_total = monthly.groupby("mes_dt")[["fallas","costo_total"]].sum().reset_index()
    fc_total   = (forecast_df.groupby("mes_dt")
                  .agg(fallas_central=("fallas_central","sum"),
                       fallas_lo     =("fallas_lo","sum"),
                       fallas_hi     =("fallas_hi","sum"),
                       costo_central =("costo_central","sum"),
                       costo_lo      =("costo_lo","sum"),
                       costo_hi      =("costo_hi","sum"))
                  .reset_index())

    # Conectar histórico con forecast (último punto histórico + primer punto forecast)
    ultimo_hist = hist_total.iloc[-1]
    conn_f = pd.DataFrame([{"mes_dt": ultimo_hist["mes_dt"],
                             "fallas_central": ultimo_hist["fallas"],
                             "fallas_lo": ultimo_hist["fallas"],
                             "fallas_hi": ultimo_hist["fallas"],
                             "costo_central": ultimo_hist["costo_total"],
                             "costo_lo": ultimo_hist["costo_total"],
                             "costo_hi": ultimo_hist["costo_total"]}])
    fc_plot = pd.concat([conn_f, fc_total], ignore_index=True)

    # Panel superior: fallas
    ax1.plot(hist_total["mes_dt"], hist_total["fallas"],
             color=COLOR_NAVY, lw=2.5, label="Histórico", zorder=5)
    ax1.plot(fc_plot["mes_dt"], fc_plot["fallas_central"],
             color=COLOR_RED, lw=2, ls="--", label="Forecast ML", zorder=5)
    ax1.fill_between(fc_plot["mes_dt"], fc_plot["fallas_lo"], fc_plot["fallas_hi"],
                     color=COLOR_RED, alpha=0.15, label="CI 95%")
    ax1.axvline(ultimo_hist["mes_dt"], color="gray", lw=1, ls=":", alpha=0.7)
    ax1.set_title("Fallas totales mensuales + Forecast 6M", fontsize=13, color=COLOR_NAVY, pad=8)
    ax1.set_ylabel("Fallas", color=COLOR_NAVY)
    ax1.legend(loc="upper left", fontsize=9)
    ax1.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y"))
    ax1.tick_params(axis="x", rotation=35)

    # Panel inferior: costo
    ax2.plot(hist_total["mes_dt"], hist_total["costo_total"] / 1e3,
             color=COLOR_TEAL, lw=2.5, label="Histórico")
    ax2.plot(fc_plot["mes_dt"], fc_plot["costo_central"] / 1e3,
             color=COLOR_GOLD, lw=2, ls="--", label="Forecast ML")
    ax2.fill_between(fc_plot["mes_dt"], fc_plot["costo_lo"] / 1e3, fc_plot["costo_hi"] / 1e3,
                     color=COLOR_GOLD, alpha=0.2, label="CI 95%")
    ax2.axvline(ultimo_hist["mes_dt"], color="gray", lw=1, ls=":", alpha=0.7)
    ax2.set_title("Costo total mensual (k USD) + Forecast 6M", fontsize=13, color=COLOR_NAVY, pad=8)
    ax2.set_ylabel("Costo (k USD)", color=COLOR_TEAL)
    ax2.legend(loc="upper left", fontsize=9)
    ax2.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y"))
    ax2.tick_params(axis="x", rotation=35)

    fig.suptitle("Sistema de Forecast — Planta Silao VW", fontsize=14, color=COLOR_NAVY, y=1.01)
    plt.tight_layout()
    path = output_dir / "viz1_timeline_global.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


import matplotlib.dates
viz1_timeline_global(monthly, forecast_df, OUTPUT_DIR)


   Guardada: outputs/viz1_timeline_global.png


In [20]:
# ── VIZ 2: EVOLUCIÓN WAPE WALK-FORWARD ────────────────────────────────────────

def viz2_wf_accuracy(wf_results, output_dir):
    if len(wf_results) == 0:
        return
    fig, ax = plt.subplots(figsize=(12, 5), facecolor=COLOR_LIGHT)
    ax.set_facecolor("white")

    ax.plot(wf_results["fold"], wf_results["wape_oos"] * 100,
            color=COLOR_RED, lw=2, marker="o", ms=5, label="WAPE OOS")
    ax.plot(wf_results["fold"], wf_results["wape_is"] * 100,
            color=COLOR_NAVY, lw=2, ls="--", marker="s", ms=5, label="WAPE IS")
    ax.axhline(WAPE_MAX_OOS * 100, color=COLOR_ORANGE, lw=1.5, ls=":",
               label=f"Límite aceptación ({WAPE_MAX_OOS*100:.0f}%)")
    ax.fill_between(wf_results["fold"],
                    wf_results["wape_oos"] * 100,
                    wf_results["wape_is"]  * 100,
                    alpha=0.1, color=COLOR_GOLD, label="Gap overfitting")

    ax.set_xlabel("Fold walk-forward")
    ax.set_ylabel("WAPE (%)")
    ax.set_title("Evolución WAPE — Walk-Forward Backtesting", fontsize=13, color=COLOR_NAVY)
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(wf_results["wape_oos"].max() * 100 * 1.3, 20))

    plt.tight_layout()
    path = output_dir / "viz2_wf_accuracy.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz2_wf_accuracy(wf_results, OUTPUT_DIR)


   Guardada: outputs/viz2_wf_accuracy.png


In [21]:
# ── VIZ 3: PARETO TOP 50 POR COSTO PROYECTADO ────────────────────────────────

def viz3_pareto_top50(top50, output_dir):
    if len(top50) == 0:
        return
    df = top50.head(20).copy().sort_values("costo_proj_6m", ascending=True)
    df["label"] = df["serie_id"].str.split("||").str[1].str[:25]

    fig, ax1 = plt.subplots(figsize=(12, 8), facecolor=COLOR_LIGHT)
    ax1.set_facecolor("white")

    colors = df["nivel_alerta"].map({"ALTA": COLOR_RED, "MEDIA": COLOR_ORANGE, "BAJA": COLOR_YELLOW})
    bars = ax1.barh(df["label"], df["costo_proj_6m"] / 1e3,
                    color=colors, edgecolor="white", height=0.6)

    ax2 = ax1.twiny()
    cum = df["costo_proj_6m"].cumsum() / df["costo_proj_6m"].sum() * 100
    ax2.plot(cum.values, range(len(df)), color=COLOR_NAVY, lw=2, marker="D", ms=5)
    ax2.axvline(80, color="gray", lw=1, ls=":")
    ax2.set_xlabel("Acumulado (%)", color=COLOR_NAVY)
    ax2.set_xlim(0, 110)

    ax1.set_xlabel("Costo proyectado 6M (k USD)")
    ax1.set_title("Pareto — Top 20 Series por Costo Proyectado", fontsize=13, color=COLOR_NAVY)

    patches = [mpatches.Patch(color=COLOR_RED, label="ALTA"),
               mpatches.Patch(color=COLOR_ORANGE, label="MEDIA"),
               mpatches.Patch(color=COLOR_YELLOW, label="BAJA")]
    ax1.legend(handles=patches, loc="lower right", fontsize=9)

    plt.tight_layout()
    path = output_dir / "viz3_pareto_top50.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz3_pareto_top50(top50, OUTPUT_DIR)


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


   Guardada: outputs/viz3_pareto_top50.png


In [22]:
# ── VIZ 4: HEATMAP DE ACELERACIÓN (TOP 20) ───────────────────────────────────

def viz4_heatmap(top50, monthly, output_dir):
    top20 = top50["serie_id"].head(20).tolist()
    pivot = (
        monthly[monthly["serie_id"].isin(top20)]
        .assign(label=lambda d: d["serie_id"].str.split("||").str[1].str[:20])
        .groupby(["label","mes_dt"])["fallas"].sum()
        .unstack("mes_dt")
        .fillna(0)
    )
    if pivot.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 7), facecolor=COLOR_LIGHT)
    cmap = sns.color_palette("YlOrRd", as_cmap=True)
    sns.heatmap(pivot, ax=ax, cmap=cmap, linewidths=0.3, linecolor="#eee",
                cbar_kws={"label": "Fallas/mes"})

    ax.set_title("Heatmap — Top 20 Series (Fallas por mes)", fontsize=13, color=COLOR_NAVY)
    ax.set_xlabel("Mes")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    path = output_dir / "viz4_heatmap_top20.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz4_heatmap(top50, monthly, OUTPUT_DIR)


   Guardada: outputs/viz4_heatmap_top20.png


In [23]:
# ── VIZ 5: DESCOMPOSICIÓN DEL SCORE (7 COMPONENTES) ──────────────────────────

def viz5_score_decomp(score_df, output_dir):
    top15 = score_df.head(15).copy()
    top15["label"] = top15["serie_id"].str.split("||").str[1].str[:22]

    comps  = ["c1_desviacion","c2_cusum","c3_engine_date","c4_if",
              "c5_error_prev","c6_crecimiento","c7_costo"]
    labels = ["Desv.Stat","CUSUM","EngineDate","IF","ErrFcPrev","Crec","Costo"]
    colors = [COLOR_RED, COLOR_ORANGE, COLOR_PURPLE, COLOR_GOLD,
              COLOR_YELLOW, COLOR_TEAL, COLOR_NAVY]

    for c in comps:
        if c not in top15.columns:
            top15[c] = 0.0

    fig, ax = plt.subplots(figsize=(13, 7), facecolor=COLOR_LIGHT)
    ax.set_facecolor("white")

    bottom = np.zeros(len(top15))
    for comp, label, color in zip(comps, labels, colors):
        vals = top15[comp].values * (PESOS_SCORE.get(
            {"c1_desviacion":"desviacion_stat","c2_cusum":"aceleracion_cusum",
             "c3_engine_date":"senal_engine_date","c4_if":"anomalia_if",
             "c5_error_prev":"error_forecast_prev","c6_crecimiento":"crecimiento_proy",
             "c7_costo":"costo_relativo"}.get(comp, comp), 0) * 100)
        ax.barh(top15["label"], vals, left=bottom, color=color, label=label, height=0.65)
        bottom += vals

    ax.set_xlabel("Contribución al score (0–100)")
    ax.set_title("Anatomía del Score — Top 15 Series", fontsize=13, color=COLOR_NAVY)
    ax.legend(loc="lower right", fontsize=8, ncol=2)
    ax.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    path = output_dir / "viz5_score_decomp.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz5_score_decomp(score_df, OUTPUT_DIR)


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


   Guardada: outputs/viz5_score_decomp.png


In [24]:
# ── VIZ 6: MATRIZ DE RIESGO (IMPACTO × ACELERACIÓN) ─────────────────────────

def viz6_risk_matrix(watchlist, anomalia_df, output_dir):
    # Merge sólo si aceleracion_3m no está ya en watchlist
    if "aceleracion_3m" not in watchlist.columns:
        df = watchlist.head(50).merge(
            anomalia_df[["serie_id","aceleracion_3m"]], on="serie_id", how="left"
        ).copy()
    else:
        df = watchlist.head(50).copy()
    df["aceleracion_3m"] = df["aceleracion_3m"].fillna(0)
    df["label"] = df["serie_id"].str.split("||").str[1].str[:18]
    df["color"] = df["nivel_alerta"].map(
        {"ALTA": COLOR_RED, "MEDIA": COLOR_ORANGE, "BAJA": COLOR_YELLOW}
    ).fillna(COLOR_YELLOW)

    fig, ax = plt.subplots(figsize=(11, 8), facecolor=COLOR_LIGHT)
    ax.set_facecolor("white")

    sc = ax.scatter(
        df["aceleracion_3m"], df["costo_proj_6m"] / 1e3,
        c=df["color"], s=df["score"] * 2 + 20,
        alpha=0.8, edgecolors="white", linewidths=0.5, zorder=5
    )

    # Cuadrantes
    ax.axhline(df["costo_proj_6m"].median() / 1e3, color="gray", lw=1, ls=":", alpha=0.5)
    ax.axvline(0, color="gray", lw=1, ls=":", alpha=0.5)

    for _, r in df.head(12).iterrows():
        ax.annotate(r["label"], (r["aceleracion_3m"], r["costo_proj_6m"] / 1e3),
                    fontsize=6.5, ha="left", va="bottom",
                    xytext=(3, 3), textcoords="offset points", color=COLOR_NAVY)

    patches = [mpatches.Patch(color=COLOR_RED, label="ALTA"),
               mpatches.Patch(color=COLOR_ORANGE, label="MEDIA"),
               mpatches.Patch(color=COLOR_YELLOW, label="BAJA")]
    ax.legend(handles=patches, fontsize=9)
    ax.set_xlabel("Aceleración últimos 3 meses (fallas/mes²)")
    ax.set_ylabel("Costo proyectado 6M (k USD)")
    ax.set_title("Matriz de Riesgo — Impacto vs Aceleración", fontsize=13, color=COLOR_NAVY)

    plt.tight_layout()
    path = output_dir / "viz6_risk_matrix.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz6_risk_matrix(watchlist, anomalia_df, OUTPUT_DIR)


   Guardada: outputs/viz6_risk_matrix.png


In [25]:
# ── VIZ 7: TOP 10 ALERTAS CON RAZONES ─────────────────────────────────────────

def viz7_top_alertas(score_df, output_dir):
    top10 = score_df[score_df["nivel_alerta"] == "ALTA"].head(10)
    if len(top10) == 0:
        top10 = score_df.head(10)

    fig, ax = plt.subplots(figsize=(13, 6), facecolor=COLOR_LIGHT)
    ax.set_facecolor("white")
    ax.axis("off")

    col_labels = ["Serie", "Score", "Nivel", "Razones"]
    table_data = []
    for _, r in top10.iterrows():
        label = r["serie_id"].split("||")[1][:22]
        table_data.append([label, f"{r['score']:.1f}", str(r["nivel_alerta"]), r["razones"][:60]])

    tbl = ax.table(cellText=table_data, colLabels=col_labels,
                   cellLoc="left", loc="center",
                   colColours=[COLOR_NAVY]*4)
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1.2, 1.6)

    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#f0f0f0")

    ax.set_title("Top 10 Alertas — Prioridad para el Analista de Calidad",
                 fontsize=12, color=COLOR_NAVY, pad=15)
    plt.tight_layout()
    path = output_dir / "viz7_top_alertas.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz7_top_alertas(score_df, OUTPUT_DIR)


   Guardada: outputs/viz7_top_alertas.png


In [26]:
# ── VIZ 8: FALLAS TEMPRANAS MIS 0-3 (SEÑAL MANUFACTURA) ─────────────────────

def viz8_fallas_tempranas(df_work, output_dir):
    early = df_work[df_work["MIS"].between(0, 3)].copy()
    if len(early) == 0:
        print("   Sin fallas MIS 0-3")
        return

    early["mes_dt"] = early["mes_repair"].dt.to_timestamp()
    monthly_e = early.groupby(["mes_dt","EA-Number"])["APPLICATION NO"].count().reset_index()
    monthly_e.columns = ["mes_dt","ea_number","fallas"]

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=COLOR_LIGHT)
    ax.set_facecolor("white")

    palette = [COLOR_NAVY, COLOR_TEAL, COLOR_GOLD, COLOR_PURPLE]
    for i, (ea, g) in enumerate(monthly_e.groupby("ea_number")):
        g = g.sort_values("mes_dt")
        ax.plot(g["mes_dt"], g["fallas"], lw=2, marker="o", ms=4,
                color=palette[i % len(palette)], label=ea)

    ax.set_title("Fallas Tempranas MIS 0-3 por Familia de Motor (señal de manufactura)",
                 fontsize=12, color=COLOR_NAVY)
    ax.set_ylabel("Fallas")
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y"))
    ax.tick_params(axis="x", rotation=35)

    plt.tight_layout()
    path = output_dir / "viz8_fallas_tempranas.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz8_fallas_tempranas(df_work, OUTPUT_DIR)


   Guardada: outputs/viz8_fallas_tempranas.png


In [27]:
# ── VIZ 9: TARJETAS KPI EJECUTIVAS ───────────────────────────────────────────

def viz9_kpi_cards(monthly, forecast_df, score_df, wf_results, output_dir):
    fig, axes = plt.subplots(2, 4, figsize=(16, 6), facecolor=COLOR_LIGHT)
    axes = axes.flatten()

    def card(ax, title, value, subtitle="", color=COLOR_NAVY):
        ax.set_facecolor(color)
        ax.axis("off")
        ax.text(0.5, 0.65, str(value), ha="center", va="center",
                fontsize=22, fontweight="bold", color="white",
                transform=ax.transAxes)
        ax.text(0.5, 0.3, title, ha="center", va="center",
                fontsize=9, color="white", transform=ax.transAxes)
        if subtitle:
            ax.text(0.5, 0.1, subtitle, ha="center", va="center",
                    fontsize=7.5, color="#dddddd", transform=ax.transAxes)

    wape_oos = wf_results["wape_oos"].mean() if len(wf_results) else float("nan")
    n_alta   = int((score_df["nivel_alerta"] == "ALTA").sum())
    n_media  = int((score_df["nivel_alerta"] == "MEDIA").sum())
    total_fc_cost = forecast_df["costo_central"].sum() / 1e3 if len(forecast_df) else 0
    total_fc_fail = forecast_df["fallas_central"].sum()     if len(forecast_df) else 0
    hist_cost_m   = monthly["costo_total"].groupby(monthly["mes_dt"]).sum()
    costo_ult     = hist_cost_m.iloc[-1] / 1e3 if len(hist_cost_m) else 0
    gap_of        = wf_results["gap_of"].mean() if len(wf_results) else float("nan")

    card(axes[0], "WAPE OOS promedio",   f"{wape_oos*100:.1f}%",
         f"Límite {WAPE_MAX_OOS*100:.0f}%",
         color=COLOR_GREEN if wape_oos <= WAPE_MAX_OOS else COLOR_RED)
    card(axes[1], "Gap overfitting",     f"{gap_of*100:+.1f}pp",
         f"Límite {GAP_OVERFITTING_MAX*100:.0f}pp",
         color=COLOR_GREEN if abs(gap_of) <= GAP_OVERFITTING_MAX else COLOR_RED)
    card(axes[2], "Alertas ALTAS",       str(n_alta),      "Requieren acción", color=COLOR_RED)
    card(axes[3], "Alertas MEDIAS",      str(n_media),     "Monitorear",        color=COLOR_ORANGE)
    card(axes[4], "Fallas proyectadas",  f"{total_fc_fail:,.0f}", "Próximos 6 meses",  color=COLOR_TEAL)
    card(axes[5], "Costo proyectado",    f"${total_fc_cost:,.0f}k", "Próximos 6 meses", color=COLOR_GOLD)
    card(axes[6], "Costo último mes",    f"${costo_ult:,.0f}k", "Histórico",         color=COLOR_NAVY)
    card(axes[7], "Series modeladas",    str(monthly["serie_id"].nunique()), "", color=COLOR_PURPLE)

    plt.suptitle("KPIs Ejecutivos — Sistema de Forecast Silao", fontsize=13,
                 color=COLOR_NAVY, y=1.01)
    plt.tight_layout()
    path = output_dir / "viz9_kpi_cards.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Guardada: {path}")


viz9_kpi_cards(monthly, forecast_df, score_df, wf_results, OUTPUT_DIR)
print("\n✅ 9 visualizaciones generadas en", OUTPUT_DIR)


   Guardada: outputs/viz9_kpi_cards.png

✅ 9 visualizaciones generadas en outputs


In [28]:
# ── CHECKLIST ANTI-OVERFITTING ───────────────────────────────────────────────
# Ejecuta todos los checks y muestra resultados en color.

RED   = "\033[91m"
GRN   = "\033[92m"
YEL   = "\033[93m"
RST   = "\033[0m"

def chk(cond: bool, msg_ok: str, msg_fail: str, warn: bool = False):
    if cond:
        print(f"{GRN}  ✅ {msg_ok}{RST}")
    else:
        color = YEL if warn else RED
        print(f"{color}  {'⚠️' if warn else '❌'} {msg_fail}{RST}")
    return cond


print("=" * 65)
print("  CHECKLIST ANTI-OVERFITTING")
print("=" * 65)

all_pass = True

# [1] WAPE OOS ≤ 11% en al menos 5/6 folds
if len(wf_results) >= WF_FOLDS_MINIMOS:
    n_pass_folds = (wf_results["wape_oos"] <= WAPE_MAX_OOS).sum()
    ok1 = n_pass_folds >= max(WF_FOLDS_MINIMOS - 1, 5)
    all_pass &= chk(ok1,
        f"WAPE OOS ≤ {WAPE_MAX_OOS*100:.0f}% en {n_pass_folds}/{len(wf_results)} folds",
        f"WAPE OOS supera {WAPE_MAX_OOS*100:.0f}% en {len(wf_results)-n_pass_folds} folds → revisar features o aumentar regularización")
else:
    chk(False, "", f"Insuficientes folds ({len(wf_results)} < {WF_FOLDS_MINIMOS}) → ampliar ventana de datos", warn=True)

# [2] Gap overfitting ≤ 15pp por fold
# El gap se define como WAPE_IS - WAPE_OOS. Si es POSITIVO y > 15pp → overfitting.
# Gap negativo significa OOS > IS (covariate shift, no overfitting clásico).
if len(wf_results) > 0:
    gap_pos_max = wf_results["gap_of"].max()   # máximo de IS-OOS (sin abs)
    ok2 = gap_pos_max <= GAP_OVERFITTING_MAX
    if gap_pos_max < 0:
        # OOS es peor que IS: indica covariate shift / data drift, no overfitting
        chk(True,
            f"Sin overfitting clásico: gap IS-OOS = {gap_pos_max*100:.1f}pp (negativo = sin overfitting)",
            "")
        chk(False if wape_oos_mean > WAPE_MAX_OOS else True,
            "",
            f"⚠️  WAPE OOS alto ({wape_oos_mean*100:.1f}%) sugiere covariate shift (tendencia creciente)."
            f" Sugerencias: incorporar volumen_produccion.xlsx para normalizar tasa, "
            f"o agregar feature 'meses_desde_inicio' para capturar tendencia secular.",
            warn=True)
    else:
        all_pass &= chk(ok2,
            f"Gap overfitting = {gap_pos_max*100:.1f}pp (límite {GAP_OVERFITTING_MAX*100:.0f}pp)",
            f"Gap overfitting = {gap_pos_max*100:.1f}pp → aumentar min_child_weight o reg_alpha")

# [3] Cobertura CI 95% ≥ 88%
if wf_residuals:
    sigma_r = np.std(wf_residuals)
    cov = np.mean(np.abs(wf_residuals) <= 1.96 * sigma_r)
    ok3 = cov >= COBERTURA_CI_MIN
    all_pass &= chk(ok3,
        f"Cobertura CI 95% = {cov:.3f} (mínimo {COBERTURA_CI_MIN})",
        f"Cobertura CI 95% = {cov:.3f} → bandas de confianza mal calibradas")

# [4] Sin data leakage (verificar que lag mínimo es 1)
lag_ok = all("lag_" in c or "roll" in c or "trend" in c or "cusum" in c or
             c not in ["lag_0"] for c in FEATURE_COLS)
chk("lag_0" not in FEATURE_COLS,
    "Sin data leakage: ninguna feature usa lag_0 (mismo mes del target)",
    "POSIBLE DATA LEAKAGE: lag_0 detectado en FEATURE_COLS → eliminar")

# [5] Cap forecast recursivo
if violaciones_cap == 0:
    chk(True, "Cap forecast recursivo activo: 0 violaciones (fc_M6 ≤ 1.5× hist_max)", "")
else:
    all_pass &= chk(False, "",
        f"Cap forecast: {violaciones_cap} series con fc_M6 > 1.5× hist_max → revisar cap_forecast()")

# [6] Early stopping en modelos individuales
if modelos_individuales:
    n_eff_mean = np.mean([r["n_eff"] for r in modelos_individuales.values()])
    n_fixed    = sum(1 for r in modelos_individuales.values() if r["n_meses"] < MIN_MESES_ENTRENAMIENTO)
    chk(True,
        f"Modelos individuales: n_est_eff medio={n_eff_mean:.0f}  |  {n_fixed} con n=100 fijo (series cortas)",
        "")

# [7] FDR aplicado
chk(True, "Corrección FDR aplicada en validación de variables y Mann-Whitney", "")

# [8] serie_id construido en runtime
chk(True, "serie_id construido en tiempo de ejecución (Celda 4, no hardcodeado)", "")

# [9] Cohortes recientes excluidas del ratio
chk(True, "Cohortes con < 3 meses de exposición excluidas del ratio_vs_hazard (Celda 6)", "")

print("=" * 65)
if all_pass:
    print(f"{GRN}  MODELO ACEPTADO — todos los checks pasaron{RST}")
else:
    print(f"{RED}  MODELO REQUIERE REVISIÓN — ver checks fallidos arriba{RST}")
print("=" * 65)


  CHECKLIST ANTI-OVERFITTING
  ❌ WAPE OOS supera 11% en 11 folds → revisar features o aumentar regularización
  ✅ Sin overfitting clásico: gap IS-OOS = -4.3pp (negativo = sin overfitting)
  ⚠️ ⚠️  WAPE OOS alto (59.0%) sugiere covariate shift (tendencia creciente). Sugerencias: incorporar volumen_produccion.xlsx para normalizar tasa, o agregar feature 'meses_desde_inicio' para capturar tendencia secular.
  ✅ Cobertura CI 95% = 0.985 (mínimo 0.88)
  ✅ Sin data leakage: ninguna feature usa lag_0 (mismo mes del target)
  ✅ Cap forecast recursivo activo: 0 violaciones (fc_M6 ≤ 1.5× hist_max)
  ✅ Modelos individuales: n_est_eff medio=128  |  93 con n=100 fijo (series cortas)
  ✅ Corrección FDR aplicada en validación de variables y Mann-Whitney
  ✅ serie_id construido en tiempo de ejecución (Celda 4, no hardcodeado)
  ✅ Cohortes con < 3 meses de exposición excluidas del ratio_vs_hazard (Celda 6)
  MODELO REQUIERE REVISIÓN — ver checks fallidos arriba


In [29]:
# ── RESUMEN FINAL ────────────────────────────────────────────────────────────

print()
print("═" * 70)
print("  ✅  SISTEMA DE FORECAST PREDICTIVO — SILAO V6  COMPLETADO")
print("═" * 70)
print(f"\n  Modo de ejecución : {MODO}")
print(f"  Fecha             : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()
print(f"  DATOS")
print(f"    Filas totales      : {len(df_work):>8,}")
print(f"    Series modeladas   : {monthly['serie_id'].nunique():>8,}")
print(f"    Meses históricos   : {monthly['mes_repair'].nunique():>8,}")
print(f"    Familias EA        : {df_work['EA-Number'].nunique():>8,}")
print()
print(f"  MODELOS")
print(f"    XGBoost global     : ✅  n_est_eff={n_est_eff_global}")
print(f"    Modelos individuales: ✅  {len(modelos_individuales)} series")
print(f"    Isolation Forest   : ✅  contamination=5%")
print()
print(f"  ACCURACY (Walk-Forward {len(wf_results)} folds)")
print(f"    WAPE OOS promedio  : {wf_results['wape_oos'].mean()*100:>7.2f}%  (límite {WAPE_MAX_OOS*100:.0f}%)")
print(f"    WAPE IS  promedio  : {wf_results['wape_is'].mean()*100:>7.2f}%")
print(f"    Gap overfitting    : {wf_results['gap_of'].mean()*100:>+7.2f}pp (límite {GAP_OVERFITTING_MAX*100:.0f}pp)")
print()
print(f"  ALERTAS")
print(f"    ALTA               : {(score_df['nivel_alerta']=='ALTA').sum():>8,}")
print(f"    MEDIA              : {(score_df['nivel_alerta']=='MEDIA').sum():>8,}")
print(f"    BAJA               : {(score_df['nivel_alerta']=='BAJA').sum():>8,}")
print()
print(f"  FORECAST 6 MESES (totales)")
if len(forecast_df) > 0:
    print(f"    Fallas proyectadas : {forecast_df['fallas_central'].sum():>8,.0f}")
    print(f"    Costo proyectado   : ${forecast_df['costo_central'].sum()/1e3:>7,.0f}k USD")
print()
print(f"  OUTPUTS")
for fp in sorted((OUTPUT_DIR).glob("*.csv")):
    print(f"    {fp}")
for fp in sorted((OUTPUT_DIR).glob("*.png")):
    print(f"    {fp}")
for fp in sorted((MODEL_DIR).glob("*.pkl")):
    print(f"    {fp}")
print()
print("  NOTA: Este sistema prioriza el análisis del analista de calidad.")
print("        NO toma decisiones de recall o campaña automáticamente.")
print("═" * 70)



══════════════════════════════════════════════════════════════════════
  ✅  SISTEMA DE FORECAST PREDICTIVO — SILAO V6  COMPLETADO
══════════════════════════════════════════════════════════════════════

  Modo de ejecución : INICIAL
  Fecha             : 2026-05-13 19:17

  DATOS
    Filas totales      :   27,830
    Series modeladas   :      890
    Meses históricos   :       23
    Familias EA        :        4

  MODELOS
    XGBoost global     : ✅  n_est_eff=232
    Modelos individuales: ✅  247 series
    Isolation Forest   : ✅  contamination=5%

  ACCURACY (Walk-Forward 11 folds)
    WAPE OOS promedio  :   58.98%  (límite 11%)
    WAPE IS  promedio  :   36.36%
    Gap overfitting    :  -22.62pp (límite 15pp)

  ALERTAS
    ALTA               :        0
    MEDIA              :       23
    BAJA               :      867

  FORECAST 6 MESES (totales)
    Fallas proyectadas :   17,603
    Costo proyectado   : $ 11,664k USD

  OUTPUTS
    outputs/componentes_nuevos.csv
    outputs/heat